# April 2025

# Run Functions

In [3]:
%run "Baseball Functions.ipynb"

# Imports and Data

In [4]:
import pandas as pd
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
import sklearn as skl
import subprocess
from moviepy.editor import VideoFileClip, concatenate_videoclips, AudioFileClip, CompositeVideoClip, TextClip
import cv2
import duckdb as db # Yegor tells me about duckdb, this will allow me to use SQL in the context of python
# claude AI, I don't remember what this was

#%run "Baseball Functions.ipynb"
woba_weights = pd.read_csv('wOBA and FIP Constants.csv')

# Load All Non-Phillies Player Data
nphl = get_nphillies_data()

# Load Phillies Player Data With Function I wrote lol
phils_sc, pos, pps = get_phillies_data()

ImportError: DLL load failed while importing cv2: The specified procedure could not be found.

In [ ]:
pps['x_break'] = pps.pfx_x.abs()
pps['z_break'] = pps.pfx_z.abs()
pos['wpa'] = pos.delta_home_win_exp.abs()
pps['wpa'] = pps.delta_home_win_exp.abs()
hits = ['single','double','triple','home_run']
swings = ['foul'
          ,'foul_bunt'
          ,'foul_tip'
          ,'hit_into_play'
          ,'missed_bunt'
          ,'swinging_pitchout'
          ,'swinging_strike'
          ,'swinging_strike_blocked'
         ]
whiffs = ['foul_tip','missed_bunt','swinging_pitchout','swinging_strike','swinging_strike_blocked']
pos['date'] = pd.to_datetime(pos.game_date)
pos['month'] = pos['date'].dt.month
pos['dow'] = pos['date'].dt.dayofweek
pps['date'] = pd.to_datetime(pps.game_date)
pps['month'] = pps['date'].dt.month
pps['dow'] = pps['date'].dt.dayofweek


# Spring Training 2025 Loading
ppst25 = pd.read_csv('spring_training_2025 ppst.csv')
post25 = pd.read_csv('spring_training_2025 post.csv')
phils_st_2025 = pd.concat([pps, pos])
st25_pps = ppst25.merge(woba_weights, left_on = 'game_year', right_on = 'Season', suffixes = ('_bad',''))
st25_pos = post25.merge(woba_weights, left_on = 'game_year', right_on = 'Season', suffixes = ('_bad',''))
st25_phils_sc = phils_st_2025.merge(woba_weights, left_on = 'game_year', right_on = 'Season', suffixes = ('_bad',''))

# LAD (H) 44-46
The defending World Series champion Los Angeles Dodgers have not lost a game in the new season. They went to Japan in mid-March and took two games over the Cubs before pummeling the Tigers and Braves at home this last week. They make the journey out to Citizens Bank Park to take on the Philadelphia Phillies who have won 5 of their first 6 games this season. After beating the division-rival Washington Nationals in the nation's capital to open the year, the Fightin Phils swept away the offensively-challenged Colorado Rockies this week.

This will be a sturdy test for two teams expected to compete for the National League pennant and World Series crown this year. The Dodgers bring their loaded roster into town while the Phillies boast one of the highest payrolls in the game with plenty of star power to boot.

## Game 1: Luzardo vs Yamamoto
This is the first time the Phillies have seen Yoshinobu Yamamoto. He pitched in the World Baseball Classic (I am fairly certain?) So I will check to see if guys faced him then ever. But I think this will truly be it. I am not sure there is anyone in baseball to compare him to so it will be interesting to see how they fare.

Luzardo comes off a pretty strong Phillies debut to make his home debut for the local nine. He will don the atrocious City Connect uniforms for this Friday night bout with the Dodgers that will appear on Apple TV. I think I need to get my booty to a bar in order to watch the game.

In [ ]:
# Ok what kinds of questions do I have about Yoshinobu Yamamoto?
# Results
op = yy = nphl[nphl.player_name == 'Yamamoto, Yoshinobu']
res_op = nresults('game_year',op)
res_op

In [ ]:
# Pitch Mix
lpm = lhb_pitch_mix(op)
lpm['stand'] = 'L'
rpm = rhb_pitch_mix(op)
rpm['stand'] = 'R'
pm = pd.concat([lpm,rpm])
fig = px.scatter(pm
                 ,x='plate_x'
                 ,y='plate_z'
                 ,color='pitch_name'
                 ,size='usage'
                 ,text='pitch_type'
                 ,title = '{} {} Pitch Mix'.format(op.player_name.unique()[0].split(', ')[1]
                                                   ,op.player_name.unique()[0].split(', ')[0]
                                                  )
                 ,facet_col = 'stand'
                )

x_axes = [ax for ax in fig.layout if ax.startswith('xaxis')]
y_axes = [ax for ax in fig.layout if ax.startswith('yaxis')]

# Add Strike Zone
fig.add_shape(type='rect'
              ,x0=-0.83,x1=0.83
              ,y0=op.sz_bot.mean()
              ,y1=op.sz_top.mean()
              ,line=dict(color='black',width=1)
              #,xref=x_ax.replace('axis', '')
              #,yref=y_ax.replace('axis', '')
             )

fig.show()

In [ ]:
# Pitches by Count
fig = px.scatter(op.sort_values(by=['balls','strikes'])
                 ,x='plate_x'
                 ,y='plate_z'
                 ,color='pitch_name'
                 ,facet_col = 'balls'
                 ,facet_row='strikes'
                )
fig.show()

In [ ]:
# Pitch Plot
def pitch_plot(df):
    fig = px.scatter(df
                     ,x='pfx_x'
                     ,y='pfx_z'
                     ,color='pitch_name'
                     ,title = '{} {} Pitch Plot'.format(df.player_name.unique()[0].split(', ')[1]
                                                        ,df.player_name.unique()[0].split(', ')[0]
                                                       )
                    )
    fig.show()

pitch_plot(op)

In [ ]:
# Yamamoto has only given up homers to RHB, this is just patently false Kellen
op[op.events == 'home_run'].groupby(['des','stand'],as_index=False).agg(hrs = ('pitch_type','count'))
# Soto, Chisholm, Stewart, and... Garrett Mitchell?
op[op.events == 'home_run'][['game_date','stand','events','inning','launch_speed','launch_angle','hit_distance_sc','pitch_type','release_speed','release_spin_rate','pfx_x','pfx_z']]
# I went ahead and included spring training in this fucking dataset

In [ ]:
# Lefty Right Splits
nresults('stand',op)
# Wow, so  Yoshinobu Yamamoto has thrown exactly the same number of pitches to lefties and righties (10 more plate appearances to RHB) and has significant reverse splits

In [ ]:
fig = px.scatter(op[(op.stand == 'L')&(op.pitch_type.isin(['FF','FS','FC','CU']))]
                 ,x='plate_x'
                 ,y='plate_z'
                 ,color='pitch_name'
                 ,title = 'Yoshinobu Yamamoto against LHB'
                )
fig.add_shape(type='rect'
              ,x0=-0.83,x1=0.83
              ,y0=op[op.stand=='L'].sz_bot.mean()
              ,y1=op[op.stand=='L'].sz_top.mean()
              ,line=dict(color='black',width=1)
             )
fig.show()

In [ ]:
# RHB are downright good against Yamamoto but LHBs really struggle, curious to see if Rob Thomson will play into that with his lineup tonight

In [ ]:
# Lineup Ingestion!
lineup = pd.DataFrame({'player_name' : ['Schwarber, Kyle'
                                        ,'Turner, Trea'
                                        ,'Harper, Bryce'
                                        ,'Bohm, Alec'
                                        ,'Kepler, Max'
                                        ,'Realmuto, J.T.'
                                        ,'Castellanos, Nick'
                                        ,'Stott, Bryson' # 'Sosa, Edmundo'
                                        ,'Marsh, Brandon' # 'Rojas, Johan'
                                       ]
                       ,'batting_order' : [1,2,3,4,5,6,7,8,9]
                       ,'fld_position' : [0
                                          ,6
                                          ,3
                                          ,5
                                          ,7
                                          ,2
                                          ,9
                                          ,4
                                          ,8
                                         ]
                      })
lineup

In [ ]:
# Opposing Pitcher Pitch Profile
t = op.groupby(['pitch_type','pitch_name']
             ,as_index=False
            ).agg(mu_velo = ('release_speed','mean')
                  ,std_velo = ('release_speed','std')
                  ,mu_spin = ('release_spin_rate','mean')
                  ,std_spin = ('release_spin_rate','std')
                  ,mu_xbreak = ('pfx_x','mean')
                  ,std_xbreak = ('pfx_x','std')
                  ,mu_zbreak = ('pfx_z','mean')
                  ,std_zbreak = ('pfx_z','std')
                 )
t['p_throws'] = op.p_throws.unique()[0]
t

In [ ]:
idk = pd.DataFrame()
df = pd.concat([pos[pos.player_name.isin(lineup.player_name)]
                ,nphl[nphl.player_name.isin(lineup.player_name)]
               ])
for pt in t.pitch_type:
    a = df[(df.p_throws == t.p_throws.unique()[0])
    &(df.pitch_type == pt)
    &(df.release_speed >= t[t.pitch_type == pt].mu_velo.unique()[0]-2*(t[t.pitch_type == pt].std_velo.unique()[0]))
    &(df.release_speed <= t[t.pitch_type == pt].mu_velo.unique()[0]+2*(t[t.pitch_type == pt].std_velo.unique()[0]))
    &(df.release_spin_rate >= t[t.pitch_type == pt].mu_spin.unique()[0]-2*(t[t.pitch_type == pt].std_spin.unique()[0]))
    &(df.release_spin_rate <= t[t.pitch_type == pt].mu_spin.unique()[0]+2*(t[t.pitch_type == pt].std_spin.unique()[0]))
    &(df.pfx_x >= t[t.pitch_type == pt].mu_xbreak.unique()[0]-2*(t[t.pitch_type == pt].std_xbreak.unique()[0]))
    &(df.pfx_x <= t[t.pitch_type == pt].mu_xbreak.unique()[0]+2*(t[t.pitch_type == pt].std_xbreak.unique()[0]))
    &(df.pfx_z >= t[t.pitch_type == pt].mu_zbreak.unique()[0]-2*(t[t.pitch_type == pt].std_zbreak.unique()[0]))
    &(df.pfx_z <= t[t.pitch_type == pt].mu_zbreak.unique()[0]+2*(t[t.pitch_type == pt].std_zbreak.unique()[0]))
    ]
    idk = pd.concat([idk,a])
nresults('player_name',idk.merge(lineup,on='player_name'))

In [ ]:
# And the indicators for the lineup against those type of hitters
inds = idk[idk.type=='X'].groupby(['player_name','batter']
                   ,as_index=False
                  ).agg(mu_ev = ('launch_speed','mean')
                        ,mu_la = ('launch_angle','mean')
                        ,mu_dist = ('hit_distance_sc','mean')
                        ,mu_xba = ('estimated_ba_using_speedangle','mean')
                        ,bip = ('des','count')
                       )
inds.round(3)

### Luzardo Preview

In [ ]:
dodgers = pd.read_csv('bdodgers_np.csv')
opp = dodgers.groupby(['player_name','batter'],as_index=False).agg(pitches = ('des','count')
                                                             ,games = ('game_pk','nunique')
                                                            )
df = phillies_pitcher = pd.concat([nphl[nphl.player_name == 'Luzardo, Jesús']
                                   ,pps[pps.player_name == 'Luzardo, Jesús']
                                  ])
opp_df = df[df.batter.isin(opp.batter)].merge(opp,on='batter',suffixes=('','_batter'))
opp_res = nresults('player_name_batter', opp_df)
opp_res.sort_values(by='ops')

In [ ]:
nresults('game_year',opp_df)

## Game 2: Nola vs Sasaki

## Game 3: Sanchez vs Glasnow

In [ ]:
# How frequently does Nick Castellanos get to 2-0?
nc = pos[pos.player_name == 'Castellanos, Nick']
nc[(nc.balls == 2)&(nc.strikes == 0)]

In [ ]:
es = pos[pos.player_name == 'Sosa, Edmundo']
res = nresults('game_year',es)
res['bbrate'] = res.walks/res.plate_apps
res

In [ ]:
pps[pps.p_throws=='L'].groupby(['player_name','pitcher'],as_index=False).agg(pitches = ('des','count')).sort_values(by='pitches',ascending=False).head(10)
cs_id = 650911
cs = pps[pps.pitcher == cs_id]
nresults('game_year',cs)

In [ ]:
z = cs[cs.events == 'home_run']
fig = px.scatter(z.sort_values(by='game_date')
                 ,x='game_date'
                 ,y='hit_distance_sc'
                 ,color='game_year'
                 ,title = 'Sanchez Home Runs'
                )
fig.show()

In [ ]:
df = cs[(cs.game_date >= '2024-04-29') & (cs.game_date <= '2024-07-04')].groupby(['game_pk','game_date']
                                                                            ,as_index=False
                                                                           ).agg(innings = ('inning','nunique')
                                                                                 ,at_bats = ('at_bat_number','nunique')
                                                                                ).sort_values(by='game_date')
df

In [ ]:
df.innings.sum()

In [ ]:
z[['game_date','inning','outs_when_up']].sort_values(by='game_date')

In [ ]:
# Does Marchan walk a lot?
po25 = pos[pos.game_year == 2025]
nresults('player_name',po25)
rm = pos[pos.player_name == 'Marchán, Rafael']
res = nresults('game_year',rm)
res['bbrate'] = res.walks/res.plate_apps
res['krate'] = res.strikeouts/res.plate_apps
res

In [ ]:
nc[(nc.on_1b.isna() == False)&(nc.on_2b.isna() == False)&(nc.on_3b.isna() == False)].groupby(['game_date'
                                                                                              ,'game_pk'
                                                                                              ,'away_team'
                                                                                              ,'home_team'
                                                                                              ,'at_bat_number'
                                                                                              ,'inning'
                                                                                              ,'events'
                                                                                             ],as_index=False
                                                                                            ).agg(pitches = ('pitch_number','max')
                                                                                                 )

# Quality ABs:
- Any plate app with 7+ piches (look at the distribution of pitches per plate_app)
- events.isin(hits)
- events.isin(['walk','hit_by_pitch','catcher_interference','intentional_walk']])
- events.isin(['sac_bunt','sac_fly'])
- Fielder's choice?
- Post_bat_score > bat_score... check this on Harper's AB against Yamamoto in 1 inn on Friday
- See 4 pitches after an 0-2 count?
- Advancing the runner to third, no I don't think so. I am not so sure to be completely honest with you.

# ATL (A) 48-410
After taking two of three from the defending World Series champion and previously unbeaten Los Angeles Dodgers, the Phillies venture out on the road to visit the division rival Atlanta Braves for an April series between the two teams expected to finish at the top of the NL East this year. The Phillies are off to a good start.

## Game 1: Wheeler vs Sale

In [ ]:
# Going to identify the Chris Sale ID through his double against Ty Kelly in 2017
pos[(pos.player_name == 'Kelly, Ty')&(pos.events == 'double')&(pos.post_bat_score == 1) & (pos.inning==8)]
sale_id = 519242

In [ ]:
# Lineup Ingestion!
lineup = pd.DataFrame({'player_name' : ['Turner, Trea'
                                        ,'Harper, Bryce'
                                        ,'Bohm, Alec'
                                        ,'Schwarber, Kyle'
                                        ,'Castellanos, Nick'
                                        ,'Kepler, Max'
                                        ,'Realmuto, J.T.'
                                        #,'Stott, Bryson' 
                                        ,'Sosa, Edmundo'
                                        #,'Marsh, Brandon' 
                                        ,'Rojas, Johan'
                                       ]
                       ,'batting_order' : [1,2,3,4,5,6,7,8,9]
                       ,'fld_position' : [6
                                          ,3
                                          ,5
                                          ,0
                                          ,9
                                          ,7
                                          ,2
                                          ,4
                                          ,8
                                         ]
                      })
lineup

In [ ]:
df = pd.concat([nphl[nphl.player_name.isin(lineup.player_name)]
                ,pos[pos.player_name.isin(lineup.player_name)]
               ]
              )
nresults('player_name',df[df.pitcher==sale_id])

In [ ]:
nresults('game_year',df[(df.player_name == 'Harper, Bryce')&(df.pitcher==sale_id)])

In [ ]:
#concat_mp4s_in_folder('Current Phillies players getting hits off Chris Sale',20)

In [ ]:
similar_hits(df[df.player_name == 'Turner, Trea']
             ,90.9,9,150,91.6)

In [ ]:
# does JT know that Ozuna does not like right on right CH/FS?
mo = pps[pps.batter == pps[pps.des.str.contains('Ozuna')].batter.unique()[0]]
nresults('game_year',mo[(mo.p_throws == 'R') & (mo.pitch_type.isin(['FS','CH']))])

In [ ]:
# WHeeler throwing a few backdoor SI
# Sale throws a lot of strikes says Rob Thomson
# Bryce Harper struggling against FF
# Schwarber hits a lot of dongs against LHP

In [ ]:
# edmundo robs a homer
similar_hits(pps
             ,102.8
             ,43
             ,352
             ,84.6
            )

In [ ]:
# jt career with 2 outs and RISP
nresults('player_name',df[(df.player_name == 'Realmuto, J.T.') & (df.outs_when_up == 2) & ((df.on_2b.isna() == False) | (df.on_3b.isna() == False))])

In [ ]:
nresults('game_year',pps[pps.player_name == 'De Los Santos, Enyel'])

In [ ]:
nresults('player_name',po25[po25.pitch_type.isin(['FF'])])

## Game 2: Walker vs Holmes

In [ ]:
# Phillies first career hits
players = lineup.player_name.tolist() + ['Marsh, Brandon'
 ,'Clemens, Kody'
 ,'Stott, Bryson'
 ,'Marchán, Rafael'
]

df = pd.concat([pos[pos.player_name.isin(players)]
                ,nphl[nphl.player_name.isin(players)]
               ])
df[df.events.isin(['single','double','triple','home_run'])].groupby('player_name'
                                                                    ,as_index=False).agg(min_game = ('game_date','min'))

In [ ]:
df.groupby('player_name'
                                                                    ,as_index=False).agg(min_game = ('game_date','min'))

In [ ]:
# Kepler 1:08:20 - 1:08:36
# Schwarber 0:39:19 - 0:39:39
# Turner 2:41:15 - 2:41:30
# Realmuto 1:14:01 - 1:14:13
# Harper 1:55:37 - 1:55:53
# Castellanos 0:00 - 0:12

# # 2-14
# hrs = 0
# mins = 0
# secs = 0
# start = (hrs*60*60)+(mins*60)+(secs*1) 
# end = start+12
# start,end

In [ ]:
# player = 'castellanos'
# hrs = 0
# mins = 0
# secs =0
# start = (hrs*60*60)+(mins*60)+(secs*1) 
# end = start+12

# subclip('Phillies first hits', '{} full'.format(player), start, end)

In [ ]:
# for f in os.listdir('C:\\Users\\Kellen\\Downloads\\Current Phillies first career hit'):
#     clip = VideoFileClip('C:\\Users\\Kellen\\Downloads\\Current Phillies first career hit\\' + f)
#     print(f, clip.w, clip.h)

In [ ]:
#concat_mp4s_in_folder('Current Phillies first career hit')

In [ ]:
df = pps[pps.game_date == '2025-04-09']

In [ ]:
# What did Taijuan Walker throw first pitch?
# A lot of Cutter and more Sinker this time (68% of his first pitches)
pitch_mix(df[(df.player_name == 'Walker, Taijuan') & (df.pitch_number == 1)])

In [ ]:
pp25 = pps[pps.game_year == 2025]
pitch_mix(pp25[(pp25.player_name == 'Walker, Taijuan') & (pp25.pitch_number == 1)])

In [ ]:
similar_hits(pd.concat([pos[pos.stand == 'R'],pps[pps.stand=='R'],nphl[nphl.stand=='R']])
             ,df[(df.hit_location == 4)&(df.inning == 5)].launch_speed.unique()[0]
             ,df[(df.hit_location == 4)&(df.inning == 5)].launch_angle.unique()[0]
             ,df[(df.hit_location == 4)&(df.inning == 5)].hit_distance_sc.unique()[0]
             ,df[(df.hit_location == 4)&(df.inning == 5)].release_speed.unique()[0]
            )

In [ ]:
nresults('player_name',df)

In [ ]:
# how frequently does a hit like Stott's RBI single in the 6thget through the infield?
similar_hits(pos[(pos.stand=='L') & (pos.hit_location.isin([3,4,9]))]
             ,pos[pos.game_date=='2025-04-09'][(pos[pos.game_date=='2025-04-09'].events == 'single')&(pos[pos.game_date=='2025-04-09'].inning == 6)&(pos[pos.game_date=='2025-04-09'].post_bat_score==1)].launch_speed.unique()[0]
             ,pos[pos.game_date=='2025-04-09'][(pos[pos.game_date=='2025-04-09'].events == 'single')&(pos[pos.game_date=='2025-04-09'].inning == 6)&(pos[pos.game_date=='2025-04-09'].post_bat_score==1)].launch_angle.unique()[0]
             ,pos[pos.game_date=='2025-04-09'][(pos[pos.game_date=='2025-04-09'].events == 'single')&(pos[pos.game_date=='2025-04-09'].inning == 6)&(pos[pos.game_date=='2025-04-09'].post_bat_score==1)].hit_distance_sc.unique()[0]
             ,pos[pos.game_date=='2025-04-09'][(pos[pos.game_date=='2025-04-09'].events == 'single')&(pos[pos.game_date=='2025-04-09'].inning == 6)&(pos[pos.game_date=='2025-04-09'].post_bat_score==1)].release_speed.unique()[0]
            )

In [ ]:
similar_hits(pos,62.2,35,183,85.7)

## Game 3: Luzardo vs Schwellenbach

In [ ]:
# Wild rain delay game and the Phils lose, bummer.

# STL (A) 411-413

In [ ]:
# Jordan Romano Fastball Velo by game
df = jr68 = romano = pp25[pp25.player_name == 'Romano, Jordan']
fig = px.box(df[df.pitch_type == 'FF']
             ,x='game_date'
             ,y='release_speed'
             ,title = 'Jordan Romano Fastball Velo by Game'
            )
fig.show()

In [ ]:
# Alec Bohm is hitting the ball hard
nresults('player_name',po25)

In [ ]:
bohm = po25[po25.player_name == 'Bohm, Alec']
po25[po25.type=='X'].groupby(['player_name','batter'],as_index=False).agg(ev = ('launch_speed','mean')
                                                          ,la = ('launch_angle','mean')
                                                          ,dist = ('hit_distance_sc','mean')
                                                         ).round(2).sort_values(by='ev',ascending=False)

In [ ]:
fig = px.scatter(bohm[bohm.type=='X']
                 ,x='launch_speed'
                 ,y='hit_distance_sc'
                 ,marginal_x = 'histogram'
                 ,marginal_y= 'histogram'
                )
fig.show()

In [ ]:
# Bohm success in STL
res = nresults('home_team',pos[pos.player_name == 'Bohm, Alec'])
res[res.home_team == 'STL']

In [ ]:
# Bohm is swinging at first pitches
res = nresults('game_year',pos[pos.player_name == 'Bohm, Alec'])
res['pitches_per_pa'] = res.pitches/res.plate_apps
res.round(3)

## Game 1: Nola vs Pallante

In [ ]:
# game preview reqs:
# take in lineup and opposing pitcher as arguments
# return opposing pitcher pitch mix and pitch plots
# return lineup results against opposing pitcher

In [ ]:
# Lineup Ingestion!
lineup = pd.DataFrame({'player_name' : ['Stott, Bryson'
                                        ,'Turner, Trea'
                                        ,'Harper, Bryce'
                                        ,'Schwarber, Kyle'
                                        ,'Castellanos, Nick'
                                        ,'Kepler, Max'
                                        ,'Bohm, Alec'
                                        ,'Marsh, Brandon'
                                        ,'Marchán, Rafael'
                                        #,'Rojas, Johan'
                                        #,'Realmuto, J.T.'
                                        #,'Sosa, Edmundo'
                                        #,'Clemens, Kody'
                                       ]
                       ,'batting_order' : [1,2,3,4,5,6,7,8,9]
                       ,'fld_position' : [4
                                          ,6
                                          ,3
                                          ,0
                                          ,9
                                          ,7
                                          ,5
                                          ,8
                                          ,2
                                         ]
                      })
lineup

def game_preview(lineup,op):
    res_op = nresults('game_year',op)
    
    # Pitch Mix
    lpm = lhb_pitch_mix(op)
    lpm['stand'] = 'L'
    rpm = rhb_pitch_mix(op)
    rpm['stand'] = 'R'
    pm = pd.concat([lpm,rpm])
    fig = px.scatter(pm
                     ,x='plate_x'
                     ,y='plate_z'
                     ,color='pitch_name'
                     ,size='usage'
                     ,text='pitch_type'
                     ,title = '{} {} Pitch Mix'.format(op.player_name.unique()[0].split(', ')[1]
                                                       ,op.player_name.unique()[0].split(', ')[0]
                                                      )
                     ,facet_col = 'stand'
                     ,hover_data = ['release_speed','release_spin_rate','pfx_x','pfx_z']
                    )
    
    x_axes = [ax for ax in fig.layout if ax.startswith('xaxis')]
    y_axes = [ax for ax in fig.layout if ax.startswith('yaxis')]
    
    # Add Strike Zone
    fig.add_shape(type='rect'
                  ,x0=-0.83,x1=0.83
                  ,y0=op.sz_bot.mean()
                  ,y1=op.sz_top.mean()
                  ,line=dict(color='black',width=1)
                  #,xref=x_ax.replace('axis', '')
                  #,yref=y_ax.replace('axis', '')
                 )
    
    fig.show()
    
    pitch_plot(op)

    # Opposing Pitcher Pitch Profile
    t = op.groupby(['pitch_type','pitch_name']
                 ,as_index=False
                ).agg(mu_velo = ('release_speed','mean')
                      ,std_velo = ('release_speed','std')
                      ,mu_spin = ('release_spin_rate','mean')
                      ,std_spin = ('release_spin_rate','std')
                      ,mu_xbreak = ('pfx_x','mean')
                      ,std_xbreak = ('pfx_x','std')
                      ,mu_zbreak = ('pfx_z','mean')
                      ,std_zbreak = ('pfx_z','std')
                     )
    t['p_throws'] = op.p_throws.unique()[0]
    #t
    
    idk = pd.DataFrame()
    df = pd.concat([pos[pos.player_name.isin(lineup.player_name)]
                    ,nphl[nphl.player_name.isin(lineup.player_name)]
                   ])
    for pt in t.pitch_type:
        a = df[(df.p_throws == t.p_throws.unique()[0])
        &(df.pitch_type == pt)
        &(df.release_speed >= t[t.pitch_type == pt].mu_velo.unique()[0]-2*(t[t.pitch_type == pt].std_velo.unique()[0]))
        &(df.release_speed <= t[t.pitch_type == pt].mu_velo.unique()[0]+2*(t[t.pitch_type == pt].std_velo.unique()[0]))
        &(df.release_spin_rate >= t[t.pitch_type == pt].mu_spin.unique()[0]-2*(t[t.pitch_type == pt].std_spin.unique()[0]))
        &(df.release_spin_rate <= t[t.pitch_type == pt].mu_spin.unique()[0]+2*(t[t.pitch_type == pt].std_spin.unique()[0]))
        &(df.pfx_x >= t[t.pitch_type == pt].mu_xbreak.unique()[0]-2*(t[t.pitch_type == pt].std_xbreak.unique()[0]))
        &(df.pfx_x <= t[t.pitch_type == pt].mu_xbreak.unique()[0]+2*(t[t.pitch_type == pt].std_xbreak.unique()[0]))
        &(df.pfx_z >= t[t.pitch_type == pt].mu_zbreak.unique()[0]-2*(t[t.pitch_type == pt].std_zbreak.unique()[0]))
        &(df.pfx_z <= t[t.pitch_type == pt].mu_zbreak.unique()[0]+2*(t[t.pitch_type == pt].std_zbreak.unique()[0]))
        ]
        idk = pd.concat([idk,a])
    res_v_op = nresults('player_name',idk.merge(lineup,on='player_name'))
    
    # And the indicators for the lineup against those type of hitters
    inds = idk[idk.type=='X'].groupby(['player_name','batter']
                       ,as_index=False
                      ).agg(mu_ev = ('launch_speed','mean')
                            ,mu_la = ('launch_angle','mean')
                            ,mu_dist = ('hit_distance_sc','mean')
                            ,mu_xba = ('estimated_ba_using_speedangle','mean')
                            ,bip = ('des','count')
                           )
    #inds.round(3)
game_preview(lineup,nphl[nphl.player_name=='Pallante, Andre'])

In [ ]:
op = nphl[nphl.player_name=='Pallante, Andre']
res_op = nresults('game_year',op)
# Opposing Pitcher Pitch Profile
t = op.groupby(['pitch_type','pitch_name']
             ,as_index=False
            ).agg(mu_velo = ('release_speed','mean')
                  ,std_velo = ('release_speed','std')
                  ,mu_spin = ('release_spin_rate','mean')
                  ,std_spin = ('release_spin_rate','std')
                  ,mu_xbreak = ('pfx_x','mean')
                  ,std_xbreak = ('pfx_x','std')
                  ,mu_zbreak = ('pfx_z','mean')
                  ,std_zbreak = ('pfx_z','std')
                 )
t['p_throws'] = op.p_throws.unique()[0]
#t

idk = pd.DataFrame()
df = pd.concat([pos[pos.player_name.isin(lineup.player_name)]
                ,nphl[nphl.player_name.isin(lineup.player_name)]
               ])
for pt in t.pitch_type:
    a = df[(df.p_throws == t.p_throws.unique()[0])
    &(df.pitch_type == pt)
    &(df.release_speed >= t[t.pitch_type == pt].mu_velo.unique()[0]-2*(t[t.pitch_type == pt].std_velo.unique()[0]))
    &(df.release_speed <= t[t.pitch_type == pt].mu_velo.unique()[0]+2*(t[t.pitch_type == pt].std_velo.unique()[0]))
    &(df.release_spin_rate >= t[t.pitch_type == pt].mu_spin.unique()[0]-2*(t[t.pitch_type == pt].std_spin.unique()[0]))
    &(df.release_spin_rate <= t[t.pitch_type == pt].mu_spin.unique()[0]+2*(t[t.pitch_type == pt].std_spin.unique()[0]))
    &(df.pfx_x >= t[t.pitch_type == pt].mu_xbreak.unique()[0]-2*(t[t.pitch_type == pt].std_xbreak.unique()[0]))
    &(df.pfx_x <= t[t.pitch_type == pt].mu_xbreak.unique()[0]+2*(t[t.pitch_type == pt].std_xbreak.unique()[0]))
    &(df.pfx_z >= t[t.pitch_type == pt].mu_zbreak.unique()[0]-2*(t[t.pitch_type == pt].std_zbreak.unique()[0]))
    &(df.pfx_z <= t[t.pitch_type == pt].mu_zbreak.unique()[0]+2*(t[t.pitch_type == pt].std_zbreak.unique()[0]))
    ]
    idk = pd.concat([idk,a])
res_v_op = nresults('player_name',idk.merge(lineup,on='player_name'))

# And the indicators for the lineup against those type of hitters
inds = idk[idk.type=='X'].groupby(['player_name','batter']
                   ,as_index=False
                  ).agg(mu_ev = ('launch_speed','mean')
                        ,mu_la = ('launch_angle','mean')
                        ,mu_dist = ('hit_distance_sc','mean')
                        ,mu_xba = ('estimated_ba_using_speedangle','mean')
                        ,bip = ('des','count')
                       )
inds.round(3)

In [ ]:
res_v_op

In [ ]:
res_op

In [ ]:
nresults('player_name',df[df.pitcher == op.pitcher.unique()[0]])

In [ ]:
# how do Phillies batters perform at STL?
nresults('player_name',df[df.home_team=='STL'])

In [ ]:
# Does Pallante have a FC?
fig = px.scatter(op[op.pitch_type == 'FF']
                 ,x='pfx_x'
                 ,y='pfx_z'
                 #,color='pfx_x'
                 ,title = 'Does Andre Pallante Have a Cutter?'
                )
fig.show()

### Ok, Nola time!

In [ ]:
nola = pps[pps.player_name == 'Nola, Aaron']
df = nola[nola.game_year == 2025]
pitch_plot(df)

In [ ]:
# Nola throwing to Marchan again, let's look at how he pitches to different catchers?
# And he issues a lead off walk, that like never happens!
r2 = nola_res_catchers = res2 = nresults('fielder_2',nola)
r2.merge(pos.groupby(['player_name','batter'],as_index=False).agg({'des' : 'count'})
         ,left_on = 'fielder_2'
         ,right_on = 'batter'
         ,suffixes = ('','_pos')
        )

### Bohm is just a disaster

In [ ]:
df = po25[po25.player_name.isin(['Marsh, Brandon','Bohm, Alec'])]
df[df.type=='X'].groupby('player_name',as_index=False).agg(ev = ('launch_speed','mean')
                                                           #,evs = ('launch_speed','std')
                                                           ,la = ('launch_angle','mean')
                                                           #,las = ('launch_angle','std')
                                                           ,dist = ('hit_distance_sc','mean')
                                                           #,dists = ('hit_distance_sc','std')
                                                          ).round(2)

### Nola cannot finish his 0-2 counts

In [ ]:
# Reaches back for 93 and blows it by the guy.
# A runner who is quick but not as quick
# What does he usually do to put guys away? Throw FF apparently?

In [ ]:
# Some wild shit happening here in the 3rd inning
# Nola walks Nootbar again (wild)
# Casty makes a nice catch and throw 

In [ ]:
# Five Strikeout Games
ks = pos[pos.events.isin(['strikeout','strikeout_double_play'])]
z = ks.groupby(['player_name'
            ,'batter'
            ,'game_pk'
            ,'game_date'
           ],as_index=False
          ).agg(punchies = ('des','count'))
z[z.punchies > 4]

In [ ]:
# Nola cannot locate his glove side SI

In [ ]:
fig = px.scatter(nola[(nola.game_year == 2025) & (nola.pitch_type == 'SI')]
                 ,x='plate_x'
                 ,y='plate_z'
                 ,color='zone'
                 ,title = 'Aaron Nola Sinkers in 2025'
                )
fig.add_shape(type='rect'
              ,x0=-0.83,x1=0.83
              ,y0=nola.sz_bot.mean(),y1=nola.sz_top.mean()
              ,line=dict(color='black',width=1)
             )
fig.show()

In [ ]:
x = nola.groupby(['bb_type','stand'],as_index=False).agg(batted_balls = ('des','count'))
x[x.stand == 'L'].batted_balls.sum(),x[x.stand == 'R'].batted_balls.sum()

In [ ]:
# First, calculate total batted balls per stand
stand_totals = x.groupby('stand')['batted_balls'].transform('sum')

# Then calculate percentage share within each stand group
x['percent_share'] = (x['batted_balls'] / stand_totals) * 100
x

In [ ]:
# Has Aaron Nola ever walked a guy with the bases loaded before? Yes just once.
nola[(nola.on_1b.isna()==False)&(nola.on_2b.isna()==False)&(nola.on_3b.isna()==False)&(nola.events == 'walk')]

In [ ]:
# What is the most number of walks Nola has had in one game?
bb = nola[nola.events=='walk']
bbs = bb.groupby(['game_pk','game_date'],as_index=False).agg(walks = ('des','count'))
bbs.sort_values(by='walks',ascending=False).head(10) # He has walked 5 guys in a game twice.

In [ ]:
fig = px.histogram(bbs,x='walks')
fig.show()

In [ ]:
fig = px.strip(bbs
               ,x='walks'
               ,orientation = 'h'
               ,stripmode = 'overlay'
              )
fig.update_traces(jitter=0.3, marker=dict(opacity=0.6, size=6))
fig.show()

### The Phillies cannot hit Andre Pallante tonight

### Jose Ruiz likes throwing to Rafael Marchan

In [ ]:
pp25.groupby('player_name',as_index=False).agg(ids = ('pitcher','max'))

In [ ]:
jr66 = ruiz = pps[pps.pitcher == 614179]
nresults('fielder_2',jr66).merge(pos.groupby('player_name',as_index=False).agg(batter = ('batter','max'))
                                 ,left_on = 'fielder_2'
                                 ,right_on='batter'
                                )

## Game 2: Sanchez vs Mikolas

In [ ]:
# How many times has Bryce Harper hit three balls 105+ mph in a game?
bh = pd.concat([nphl[nphl.player_name == 'Harper, Bryce']
                ,pos[pos.player_name == 'Harper, Bryce']
               ])
df = bh[(bh.launch_speed >= 105) & (bh.type == 'X')]
df.groupby(['game_pk','game_date','away_team','home_team'],as_index=False
          ).agg(hard_hits = ('des','count')
                ,atbats = ('at_bat_number','nunique')
               ).sort_values(by='hard_hits',ascending=False).head(20)

In [ ]:
# Most GIDPs in a game for Phillies pitchers
gidps = pps[pps.events == 'grounded_into_double_play']
gidps.groupby(['game_pk'
               ,'game_date','away_team','home_team'],as_index=False
             ).agg(dps = ('des','count')
                   ,atbats = ('at_bat_number','nunique')
                  ).sort_values(by='dps',ascending=False).head(20)

In [ ]:
# Most GIDPs in a game for a single Phillies pitcher
gidps = pps[pps.events == 'grounded_into_double_play']
gidps.groupby(['game_pk','player_name'
               ,'game_date','away_team','home_team'],as_index=False
             ).agg(dps = ('des','count')
                   ,atbats = ('at_bat_number','nunique')
                  ).sort_values(by='dps',ascending=False).head(20)

In [ ]:
# Kerk as the janitor
# Hoffman as the janitor

In [ ]:
ok = pps[pps.player_name == 'Kerkering, Orion'] # Give me Orion Kerkering Pitches
okg = ok.groupby(['game_pk','game_date'],as_index=False).agg(min_ab = ('at_bat_number','min')) # Give me his first at bat by game
df = pps[pps.pitch_number == 1].merge(okg # Filter to just the first pitch at the bat to maintain cardinality of sets!
               ,left_on = ['game_pk','game_date','at_bat_number'] # Join on the at bat and game
               ,right_on = ['game_pk','game_date','min_ab'] # To the game and the min at bat from Kerkering
               ,suffixes = ('','_okg') # Suffixes in case anything overlaps
              )
df[['game_date','away_team','home_team','inning','outs_when_up','on_1b','on_2b','on_3b']].sort_values(by='game_date'
                                                                                                      ,ascending=False
                                                                                                      ).head(15)

In [ ]:
# Sanchie walk rate
res = nresults('game_year',cs)
res['bbrate'] = res.walks/res.plate_apps
res

In [ ]:
# Who is Roddery Munoz?
# A punchout for JT and Bohm kinda guy apparently!

In [ ]:
# Psycho Romano ready to come in to the game now

## Game 3: Wheeler vs Liberatore

# SFG (H) 414-417
The Giants are good and they come to town with the chance to test themselves against a team who should be a legitimate contender in the National League. Despite losing both series on their recent road trip, the Phillies are a good team, and one that has played well at home in the last three seasons.

In [ ]:
off = pos[pos.home_team == 'PHI'].groupby(['game_pk','game_year','game_date','away_team','home_team'],as_index=False
                                         ).agg(runs_for = ('post_home_score','max')
                                              )
against = pps[pps.home_team == 'PHI'].groupby(['game_pk','game_year','game_date','away_team','home_team']
                                              ,as_index=False
                                             ).agg(runs_against = ('post_away_score','max')
                                                  )

df = off.merge(against
               ,on=['game_pk','game_date','away_team','home_team','game_year']
               ,suffixes=('_for','_against')
              )
df[df.game_year > 2021]

In [ ]:
df['wl'] = np.where(df.runs_for > df.runs_against, 'W','L')
df.groupby('game_year',as_index=False).agg(runs_for = ('runs_for','sum')
                                           ,runs_against = ('runs_against','sum')
                                          )

In [ ]:
ws = df[df.wl == 'W'].groupby('game_year',as_index=False).agg(wins = ('wl','count'))
ls = df[df.wl == 'L'].groupby('game_year',as_index=False).agg(losses = ('wl','count'))
z = ws.merge(ls,on='game_year')
z['win_pct'] = z.wins/(z.wins+z.losses)
z

## Game 1: Walker vs Roupp

In [ ]:
# Compare Tai Walker pitch plots over the last few years
tw = pps[pps.player_name == 'Walker, Taijuan']
fig = px.scatter(tw.sort_values(by='game_year')
                 ,x='pfx_x'
                 ,y='pfx_z'
                 ,color='pitch_name'
                 ,facet_col='game_year'
                )
fig.show()

In [ ]:
#concat_mp4s_in_folder('Taijuan Walker picking guys off',10)

In [ ]:
res = nresults('game_year',pos)
res['bbrate'] = res.walks/res.plate_apps
res['krate'] = res.strikeouts/res.plate_apps
res

# Chase and Whiff Rate
Could add these to the results function by enhancing for a generic level
<br> Also could improve the results function by adding K% and BB%

In [ ]:
def whiff_rate(level,df):
    swings = ['foul'
          ,'foul_bunt'
          ,'foul_tip'
          ,'hit_into_play'
          ,'missed_bunt'
          ,'swinging_pitchout'
          ,'swinging_strike'
          ,'swinging_strike_blocked'
         ]
    whiffs = ['foul_tip','missed_bunt','swinging_pitchout','swinging_strike','swinging_strike_blocked']
    u = df[df.description.isin(swings)].groupby(level,as_index=False).agg(swings = ('des','count'))
    v = df[df.description.isin(whiffs)].groupby(level,as_index=False).agg(whiffs = ('des','count'))
    w = u.merge(v,on=level)
    w['whiff_rate'] = w.whiffs/w.swings
    return w

In [ ]:
def chase_rate(level,df):
    swings = ['foul'
          ,'foul_bunt'
          ,'foul_tip'
          ,'hit_into_play'
          ,'missed_bunt'
          ,'swinging_pitchout'
          ,'swinging_strike'
          ,'swinging_strike_blocked'
         ]
    chase = df[(df.zone>9)&(df.description.isin(swings))]
    i = chase.groupby(level,as_index=False).agg(chases = ('des','count'))
    j = df[df.zone>9].groupby(level,as_index=False).agg(ooz = ('des','count'))
    cr = i.merge(j,on=level)
    cr['chase_rate'] = cr.chases/cr.ooz
    return cr

In [ ]:
# Chase Rate and Whiff Rate?
swings = ['foul'
          ,'foul_bunt'
          ,'foul_tip'
          ,'hit_into_play'
          ,'missed_bunt'
          ,'swinging_pitchout'
          ,'swinging_strike'
          ,'swinging_strike_blocked'
         ]
whiffs = ['foul_tip','missed_bunt','swinging_pitchout','swinging_strike','swinging_strike_blocked']

chase = pos[(pos.zone>9)&(pos.description.isin(swings))]
i = chase.groupby('game_year',as_index=False).agg(chases = ('des','count'))
j = pos[pos.zone>9].groupby('game_year',as_index=False).agg(ooz = ('des','count'))
k = i.merge(j,on='game_year')
k['chase_rate'] = k.chases/k.ooz
cr = k
cr

u = pos[pos.description.isin(swings)].groupby('game_year',as_index=False).agg(swing = ('des','count'))
v = pos[pos.description.isin(whiffs)].groupby('game_year',as_index=False).agg(whiff = ('des','count'))
w = u.merge(v,on='game_year')
w['whiff_rate'] = w.whiff/w.swing
w

res.merge(cr,on='game_year').merge(w,on='game_year')

In [ ]:
# RISP has to be down
# And it is, the team is hitting .204 with RISP so far this year.
# I also did a share calculation to see how many of the plate appearances they have had this year come with RISP, it is up to 27.4% which would be the highest in a season.
# Making outs with RISP increases the number of plate appearances with RISP

In [ ]:
risp = pos[(pos.on_2b.isna() == False) | (pos.on_3b.isna() == False)] # Filter to runners in scoring position
res = nresults('game_year',risp) # Get the results with RISP
gy = nresults('game_year',pos) # Get the results for the entire year, used to calculate the share of plate_apps w RISP
res = res.merge(gy,on=['game_year'],suffixes = ('','_nrisp')) # Merge the two result sets
res['share'] = round(100*(res.plate_apps/res.plate_apps_nrisp),1) # Do the share calculation and format since it is a percentage
res[[c for c in res.columns if 'nrisp' not in c]] # Give me only the columns I care about, this was cool!

In [ ]:
fig = px.scatter(nresults('player_name',risp[risp.game_year == 2025])
                 ,x='plate_apps'
                 ,y='ba'
                 ,size='pitches'
                 ,title = 'Batters with RISP in 2025'
                 ,text='player_name'
                )
fig.show()
# Besides Stott, Harper, and Castellanos the rest* of the regular Phillies are at or below the Mendoza line (.200 BA) with runners in scoring position this year. 
# *Trea Turner is hitting .214 with RISP this year.

In [ ]:
# Runner on third and less than two outs
df = pos[(pos.on_3b.isna() == False) & (pos.outs_when_up < 2)]
nresults('game_year',df)
# The team is hitting .286 in this situation which is the lowest batting average in the Statcast era, the previous low was .288 across 287 plate appearances in 2017.

In [ ]:
atbats = df.groupby(['game_date'
                                           ,'game_year'
                                  ,'game_pk'
                                  ,'at_bat_number'
                                  ,'inning'
                                  ,'outs_when_up'
                                  ,'player_name'
                                  ,'away_team'
                                  ,'home_team'
                                  ,'events'
                                  ,'bat_score','post_bat_score'
                                 ],as_index=False
                                ).agg(pitches = ('pitch_number','max')
                                     )
atbats['runs_scored'] = atbats.post_bat_score - atbats.bat_score
atbats # This is a list of all the plate appearances the Phillies have had this year with a runner on third and less than two outs

In [ ]:
atbats.groupby('events',as_index=False).agg(count = ('pitches','count')
                                            ,runs = ('runs_scored','sum')
                                           )

In [ ]:
atbats[atbats.player_name.isin(['Bohm, Alec','Turner, Trea','Realmuto, J.T.'])].groupby(['player_name','game_year'],as_index=False).agg(count = ('pitches','count')
                                            ,runs = ('runs_scored','sum')
                                           ).sort_values(by=['player_name','runs','count'],ascending=False)
# Bohm, Kepler, and Turner have each had five plate appearances in this situation yet have only produced one run
# Marsh has brought home two runs in his five plate appearances.
# JT and Stott have each only produced one run in their respective three plate appearances

In [ ]:
atbats.sort_values(by=['player_name','game_date'])

In [ ]:
# Feast or famine with RISP
# I don't totally know how to look at this.

In [ ]:
gms = risp[risp.game_year == 2025].groupby(['game_date','game_pk'
                                      ,'away_team','home_team'
                                            #,'post_bat_score','bat_score'
                                     ],as_index=False
                                    ).agg(at_bats = ('at_bat_number','nunique')
                                         )
gms

In [ ]:
# L/R splits for Brandon Marsh and Phillies in general
z = pd.DataFrame()
for s in ['L','R']:
    x = nresults('player_name',po25[po25.p_throws == s])
    x['p_throws'] = s
    z = pd.concat([z,x])
z.sort_values(by=['player_name','p_throws'])
# Bohm struggles against everyone, slight platoon advantage
# Casty hitting lefties really well
# Harper traditional struggles, curious how it compares to his career
# Kepler traditional struggles, curious how it compares to his career
# Marsh truly truly atrocious against RHP and just normal struggles against lefties
# JT cannot hit a lefty to save his life so far
# Rojas traditional platoon splits
# Kyle Schwarber mashing against lefties
# Edmundo is red hot against righties and very good against lefties
# Stott is atrocious against LHP at the moment, way worse than normal.
# Trea minor platoon advantage but no slug against lefties

In [ ]:
# Phillies not going the opposite way
# I just don't know how to interpret these location data
# The spray charts just don't do it for me

In [ ]:
# Just going to try building a spray chart for RHB nad LHB of the Philadelphia Phillies
fig = px.scatter(po25[po25.stand == 'R']
                 ,x='hc_x'
                 ,y='hc_y'
                 ,color='hit_location'
                 ,hover_data = ['player_name','game_date','inning','events']
                 ,title = 'RHB Spray Chart'
                )
fig.show()

In [ ]:
# Just going to try building a spray chart for RHB nad LHB of the Philadelphia Phillies
fig = px.scatter(po25[po25.stand == 'L']
                 ,x='hc_x'
                 ,y='hc_y'
                 ,color='hit_location'
                 ,hover_data = ['player_name','game_date','inning','events']
                 ,title = 'LHB Spray Chart'
                )
fig.show()

In [ ]:
# Do the Phillies use the platoon advantage historically?
# Just look at the share of pitches thrown LvR vs Lvl and RvR

In [ ]:
j = pd.DataFrame()
for t,s in zip(['L','L','R','R'],['L','R','L','R']):
    print(t,s)
    i = nresults('game_year',pos[(pos.p_throws == t) & (pos.stand == s)])
    i['p_throws'] = t
    i['stand'] = s
    j = pd.concat([j,i])
j[j.p_throws != j.stand].sort_values(by=['game_year','p_throws','stand'])

In [ ]:
# Alec Bohm hard hit balls in 2025
df = bohm = pos[pos.player_name == 'Bohm, Alec']
level = 'game_year'
def hard_hits(level,df):
    bip = df[df.type == 'X'].groupby(level,as_index=False).agg(balls_in_play = ('des','count')
                                                               ,ev_mu = ('launch_speed','mean')
                                                               ,la_mu = ('launch_angle','mean')
                                                               ,dist_mu = ('hit_distance_sc','mean')
                                                              )
    hh = df[(df.type == 'X') & (df.launch_speed >= 95)].groupby(level,as_index=False).agg(hard_hits = ('des','count')
                                                                       ,ev_mu = ('launch_speed','mean')
                                                                       ,la_mu = ('launch_angle','mean')
                                                                       ,dist_mu = ('hit_distance_sc','mean')
                                                                      )
    z = bip.merge(hh, on = level, suffixes = ('','_hh')).round(1)
    z['hard_hit_pct'] = round(100*(z.hard_hits/z.balls_in_play),1)
    return z[[c for c in z.columns if '_hh' not in c]].sort_values(by='hard_hit_pct',ascending=False)
hard_hits('game_year',bohm)

In [ ]:
# How about Hard Hit % for all Phillies players?
hard_hits('player_name',po25)

In [ ]:
hard_hits(['player_name','game_year'],pos[pos.game_year >= 2022]).head(20) # so nobody hits more than 50% of their batted balls 95+ mph unless your name is Kyle Schwarber

## Game 2: Luzardo vs Verlander

In [ ]:
# Lineup Ingestion!
lineup = pd.DataFrame({'player_name' : ['Stott, Bryson'
                                        ,'Turner, Trea'
                                        ,'Harper, Bryce'
                                        ,'Schwarber, Kyle'
                                        ,'Castellanos, Nick'
                                        ,'Realmuto, J.T.'
                                        ,'Kepler, Max'
                                        ,'Bohm, Alec'
                                        #,'Stott, Bryson' 
                                        #,'Sosa, Edmundo'
                                        #,'Marsh, Brandon' 
                                        ,'Rojas, Johan'
                                       ]
                       ,'batting_order' : [1,2,3,4,5,6,7,8,9]
                       ,'fld_position' : [4
                                          ,6
                                          ,3
                                          ,0
                                          ,9
                                          ,2
                                          ,7
                                          ,5
                                          ,8
                                         ]
                      })
lineup
df = pd.concat([nphl[nphl.player_name.isin(lineup.player_name)]
                ,pos[pos.player_name.isin(lineup.player_name)]
               ])
nresults('player_name',df[df.pitcher == nphl[nphl.player_name == 'Verlander, Justin'].pitcher.unique()[0]])

In [ ]:
pos['days'] = pos.game_date.astype('str').str.split('-').str[2]
rf = pd.concat([pos[(pos.month == 4) & (pos.days == '15')]
                ,pos[pos.game_date == '2020-08-28']
                ,pos[pos.game_date == '2021-04-16']
               ]).groupby(['game_date'
                                                   ,'game_pk'
                                                    ,'game_year'
                                                    ,'away_team'
                                                    ,'home_team'
                                                   ],as_index=False
                                                  ).agg(runs_for = ('post_bat_score','max')
                                                       )
ra = pps[pps.game_pk.isin(rf.game_pk)].groupby('game_pk',as_index=False).agg(runs_against = ('post_bat_score','max'))
jr42 = rf.merge(ra,on='game_pk')
jr42

## Game 3: Nola vs Ray

In [ ]:
# Lineup Ingestion!
lineup = pd.DataFrame({'player_name' : ['Turner, Trea'
                                        ,'Harper, Bryce'
                                        ,'Schwarber, Kyle'
                                        ,'Castellanos, Nick'
                                        ,'Realmuto, J.T.'
                                        ,'Bohm, Alec' 
                                        ,'Sosa, Edmundo'
                                        ,'Marchán, Rafael' 
                                        ,'Rojas, Johan'
                                       ]
                       ,'batting_order' : [1,2,3,4,5,6,7,8,9]
                       ,'fld_position' : [6
                                          ,3
                                          ,7
                                          ,9
                                          ,0
                                          ,5
                                          ,4
                                          ,2
                                          ,8
                                         ]
                      })
lineup
df = pd.concat([nphl[nphl.player_name.isin(lineup.player_name)]
                ,pos[pos.player_name.isin(lineup.player_name)]
               ])
nresults('player_name',df[df.pitcher == nphl[nphl.player_name == 'Ray, Robbie'].pitcher.unique()[0]])

In [ ]:
op = nphl[nphl.player_name == 'Ray, Robbie']
lpm = lhb_pitch_mix(op)
lpm['stand'] = 'L'
rpm = rhb_pitch_mix(op)
rpm['stand'] = 'R'
pm = pd.concat([lpm,rpm])
fig = px.scatter(pm
                 ,x='plate_x'
                 ,y='plate_z'
                 ,color='pitch_name'
                 ,size='usage'
                 ,text='pitch_type'
                 ,title = '{} {} Pitch Mix'.format(op.player_name.unique()[0].split(', ')[1]
                                                   ,op.player_name.unique()[0].split(', ')[0]
                                                  )
                 ,facet_col = 'stand'
                 ,hover_data = ['release_speed','release_spin_rate','pfx_x','pfx_z']
                )

x_axes = [ax for ax in fig.layout if ax.startswith('xaxis')]
y_axes = [ax for ax in fig.layout if ax.startswith('yaxis')]

# Add Strike Zone
fig.add_shape(type='rect'
              ,x0=-0.83,x1=0.83
              ,y0=op.sz_bot.mean()
              ,y1=op.sz_top.mean()
              ,line=dict(color='black',width=1)
              #,xref=x_ax.replace('axis', '')
              #,yref=y_ax.replace('axis', '')
             )

fig.show()

In [ ]:
pitch_plot(op)

### Nola Preview

In [ ]:
# Pitching to Marchan again even though there is not much evidence of that being a good thing

In [ ]:
df = nola[nola.game_year == 2025]
lpm = lhb_pitch_mix(df)
lpm['stand'] = 'L'
rpm = rhb_pitch_mix(df)
rpm['stand'] = 'R'
lpm

fig = px.scatter(pd.concat([lpm,rpm])
                 ,x='plate_x'
                 ,y='plate_z'
                 ,color='pitch_name'
                 ,size='usage'
                 ,hover_data = ['release_speed','release_spin_rate','pfx_x','pfx_z']
                 ,title = 'Aaron Nola Approach'
                 ,text='pitch_type'
                 ,facet_col = 'stand'
                )
fig.show()

In [ ]:
fig = px.box(df
                 ,x='game_date'
                 ,y='release_speed'
                 ,facet_col = 'pitch_type'
             #,facet_row = 'game_date'
                 ,color='pitch_name'
             ,title = 'Aaron Nola Pitch Velocity by Start'
                )
fig.show()

In [ ]:
# Outfield Defense

In [ ]:
# ehh idk

In [ ]:
# Max Kepler hitting the ball hard but not in clutch situations
# Win Probabiltiy added?
# Trea Turner also not coming up in the clutch perhaps

## Game 4: Sanchez vs Hicks

In [ ]:
# Sanchez second and third time through the order
min_abs = cs.groupby(['game_pk','batter'],as_index=False).agg(min_ab = ('at_bat_number','min')) # Get the minimum at bat number for each batter in each game against Sanchez

In [ ]:
merged = cs.merge(min_abs,on=['game_pk','batter']) # Merge on the game and batter
len(cs),len(merged)

In [ ]:
nresults('player_name',merged[merged.min_ab != merged.at_bat_number]) # Results when it is not the first at bat

In [ ]:
nresults('player_name',merged[merged.min_ab == merged.at_bat_number]) # Results when it is the first at bat

In [ ]:
# There does not appear to be much of a difference

In [ ]:
# Going to try one more thing - Can I add a counter that adds one each time the at_bat_number changes and resets each time the batter changes
# cs.sort_values(by=['game_pk','batter','at_bat_number'])
cs['rk'] = cs.sort_values(by=['game_pk','batter','at_bat_number']).groupby(['game_pk', 'batter'])['at_bat_number'].rank(method='dense').astype(int)
# This seemed to do it, thanks chatgpt!

In [ ]:
cs[(cs.game_date == cs.game_date.max()) & (cs.batter == 680977)][['game_pk','batter','at_bat_number','rk']]

In [ ]:
nresults('rk',cs) # So these are the results for Cristopher Sanchez based on each time through the order, he tends to improve!

In [ ]:
# Sanchez velo throughout the year
fig = px.box(cs[cs.game_year == 2024]
             ,x='game_date'
             ,y='release_speed'
             ,facet_col = 'pitch_type'
             #,facet_row = 'game_year'
             ,color='pitch_name'
             ,title = 'Cristopher Sanchez Pitch Velocity by Start'
                )

# Calculate medians per game_date and pitch_type
medians = (
    cs[cs.game_year == 2024]
    .groupby(['game_date', 'pitch_type'])['release_speed']
    .median()
    .reset_index()
    .rename(columns={'release_speed': 'median_speed'})
)

# Add a line to each facet
for pitch in medians['pitch_type'].unique():
    df = medians[medians['pitch_type'] == pitch].sort_values('game_date')
    fig.add_trace(
        go.Scatter(
            x=df['game_date'],
            y=df['median_speed'],
            mode='lines',
            name=f'{pitch} Median',
            #line=dict(dash='dash'),
            showlegend=False  # change to True if you want it in the legend
        ),
        row=1,
        col=list(medians['pitch_type'].unique()).index(pitch) + 1
    )
    
fig.show()

In [ ]:
# Sanchez velo throughout the year
fig = px.box(cs[cs.game_year == 2023]
             ,x='game_date'
             ,y='release_speed'
             ,facet_col = 'pitch_type'
             #,facet_row = 'game_year'
             ,color='pitch_name'
             ,title = 'Cristopher Sanchez Pitch Velocity by Start'
                )

# # Calculate medians per game_date and pitch_type
# medians = (
#     cs[cs.game_year == 2023]
#     .groupby(['game_date', 'pitch_type'])['release_speed']
#     .median()
#     .reset_index()
#     .rename(columns={'release_speed': 'median_speed'})
# )

# # Add a line to each facet
# for pitch in medians['pitch_type'].unique():
#     df = medians[medians['pitch_type'] == pitch].sort_values('game_date')
#     fig.add_trace(
#         go.Scatter(
#             x=df['game_date'],
#             y=df['median_speed'],
#             mode='lines',
#             name=f'{pitch} Median',
#             #line=dict(dash='dash'),
#             showlegend=False  # change to True if you want it in the legend
#         ),
#         row=1,
#         col=list(medians['pitch_type'].unique()).index(pitch) + 1
#     )
    
fig.show()

In [ ]:
# Sanchez last 10 road starts
cs[cs.away_team == 'PHI'].groupby(['game_pk' # group at the game level
                                    ,'game_date'
                                    ,'away_team' # and pull in the teams just for fun
                                    ,'home_team'
                                   ],as_index=False
                                  ).agg(max_inning = ('inning','max') # what inning he pitched into
                                        ,runs_allowed = ('post_bat_score','max') # the score when he left the game (not earned runs unfortunately)
                                        # ,mu_velo = ('release_speed','mean')
                                        # ,mu_spin = ('release_spin_rate','mean')
                                        # ,mu_xbreak = ('x_break','mean')
                                        # ,mu_zbreak = ('z_break','mean')
                                        # ,mu_ev = ('launch_speed','mean')
                                        # ,mu_la = ('launch_angle','mean')
                                        # ,mu_dist = ('hit_distance_sc','mean')
                                       ).merge(nresults('game_pk',cs) # run the results of 
                                               ,on='game_pk'
                                               ,suffixes=('','_res')
                                              ).sort_values(by='game_date' # sort to get the most recent games
                                                            ,ascending=False
                                                           ).head(10)

In [ ]:
# Sanchez last 10 home starts
cs[cs.home_team == 'PHI'].groupby(['game_pk' # group at the game level
                                    ,'game_date'
                                    ,'away_team' # and pull in the teams just for fun
                                    ,'home_team'
                                   ],as_index=False
                                  ).agg(max_inning = ('inning','max') # what inning he pitched into
                                        ,runs_allowed = ('post_bat_score','max') # the score when he left the game (not earned runs unfortunately)
                                        # ,mu_velo = ('release_speed','mean')
                                        # ,mu_spin = ('release_spin_rate','mean')
                                        # ,mu_xbreak = ('x_break','mean')
                                        # ,mu_zbreak = ('z_break','mean')
                                        # ,mu_ev = ('launch_speed','mean')
                                        # ,mu_la = ('launch_angle','mean')
                                        # ,mu_dist = ('hit_distance_sc','mean')
                                       ).merge(nresults('game_pk',cs) # run the results of 
                                               ,on='game_pk'
                                               ,suffixes=('','_res')
                                              ).sort_values(by='game_date' # sort to get the most recent games
                                                            ,ascending=False
                                                           ).head(10)

In [ ]:
# Cris Sanchez whiff rate by pitch

In [ ]:
# Whiff Rate
swings = ['foul'
          ,'foul_bunt'
          ,'foul_tip'
          ,'hit_into_play'
          ,'missed_bunt'
          ,'swinging_pitchout'
          ,'swinging_strike'
          ,'swinging_strike_blocked'
         ]
whiffs = ['foul_tip','missed_bunt','swinging_pitchout','swinging_strike','swinging_strike_blocked']

# chase = pos[(pos.zone>9)&(pos.description.isin(swings))]
# i = chase.groupby('game_year',as_index=False).agg(chases = ('des','count'))
# j = pos[pos.zone>9].groupby('game_year',as_index=False).agg(ooz = ('des','count'))
# k = i.merge(j,on='game_year')
# k['chase_rate'] = k.chases/k.ooz
# cr = k
# cr
level = 'pitch_type'
df = cs[cs.game_year == 2025]
u = df[df.description.isin(swings)].groupby(level,as_index=False).agg(swing = ('des','count'))
v = df[df.description.isin(whiffs)].groupby(level,as_index=False).agg(whiff = ('des','count'))
w = u.merge(v,on=level)
w['whiff_rate'] = w.whiff/w.swing
w

#res.merge(cr,on='game_year').merge(w,on='game_year')

### Phillies Offense

In [ ]:
xbh = ['double','triple','home_run']
level = 'game_year'
xbhs = pos[pos.events.isin(xbh)].groupby(level,as_index=False).agg(xbh = ('des','count'))
all_hits = pos[pos.events.isin(hits)].groupby(level,as_index=False).agg(hits = ('des','count'))
z = xbhs.merge(all_hits,on=level)
z['xbh_pct'] = z.xbh/z.hits
z

In [ ]:
zz = z.merge(pos.groupby('game_year',as_index=False).agg(games = ('game_pk','nunique'))
        ,on='game_year'
       )
zz['p162'] = (zz.hits/zz.games)*162
zz['x162'] = (zz.xbh/zz.games)*162
zz

In [ ]:
pitch_plot(cs)

In [ ]:
po25[['game_date','away_team','home_team','pitch_type','release_speed','release_spin_rate']].sort_values(by='release_speed',ascending=False).head(10)

In [ ]:
nresults('game_year',nphl[nphl.player_name == 'Hicks, Jordan'])
op = nphl[nphl.player_name == 'Hicks, Jordan']
pitch_plot(op)

# Trea Turner Defense

In [ ]:
# Trying to watch all his plays this year since he is +2 OAA at the moment

In [ ]:
# He referenced this catch that he made in the ninth inning in Philadelphia and crashed into Howie Kendrick in either 2018 or 2019
# I just can't find it!

In [ ]:
ids = [pos[pos.player_name == 'Turner, Trea'].batter.unique()[0], pos[pos.player_name == 'Kendrick, Howie'].batter.unique()[0]]
ids

In [ ]:
pos[(pos.fielder_6 == ids[0])#&(pos.fielder_7 == ids[1])
#&(pos.home_team == 'PHI') & (pos.away_team == 'WSH')
#&(pos.bb_type == 'popup')
&(pos.bb_type != 'ground_ball')
#&(pos.hc_x>= 75) & (pos.hc_x <= 85)
#&(pos.hc_y>= 120) & (pos.hc_y <= 130)
&(pos.hit_location == 6)][['game_year'
                                                    ,'game_date'
                                                    ,'fielder_7'
                                                   ,'inning'
                                                   ,'player_name'
                                                    ,'away_team'
                                                    ,'home_team'
                                                   ,'events']]

In [ ]:
similar_hits(pps
             ,69.9,41,215,86)

In [ ]:
pos.groupby('bb_type',as_index=False).agg(count = ('des','count')).sort_values(by='count',ascending=False)

```sql
select bb_type, count(1)
from pos
group by bb_type
order by count(1) desc

```

# Phillies Pitchers First Career Strikeout

In [ ]:
# I could also do the first career strikeout for each Phillies pitcher
pitchers = pp25.groupby('player_name',as_index=False).agg(pitches = ('des','count'))
df = pd.concat([nphl[nphl.player_name.isin(pitchers.player_name)]
                ,pps[pps.player_name.isin(pitchers.player_name)]
               ])
ks = df[df.events.isin(['strikeout','strikeout_double_play'])]
ks.groupby(['player_name','pitcher'],as_index=False).agg(min_date = ('game_date','min')).sort_values(by='min_date')
# Taijuan Walker 42:20 - https://www.youtube.com/watch?v=sBydFjenKi8
# Zack Wheeler 15:20 - https://www.youtube.com/watch?v=wcCOa7_SFN8
# Zack Wheeler 0:15 - https://www.youtube.com/watch?v=wOgac84p0uQ
# Joe Ross 7:50 - https://www.youtube.com/watch?v=tK475ByJtc8
# Aaron Nola 9:10 - https://www.youtube.com/watch?v=Pw4inrNtLSE

In [ ]:
pps[pps.p_throws=='L'].groupby(['player_name','pitcher'],as_index=False).agg(pitches = ('des','count')
                                                                            ).sort_values(by='pitches',ascending=False)
rs = pps[(pps.pitcher == 624133) & (pps.events.isin(['strikeout','strikeout_double_play']))].agg(min_date = ('game_date','min'))
rs

In [ ]:
# player = 'wheeler'
# hrs = 0
# mins = 0
# secs = 15
# start = (hrs*60*60)+(mins*60)+(secs*1) 
# end = start+7

# subclip('First Ks', '{}'.format(player), start, end)

# normalize_clips('First Ks')

# concat_mp4s_in_folder('Phillies pitchers first career strikeout')

# MIA (H) 418-420
The Phillies welcome the Marlins for a three-game setthis weekend. This feels like a sweep, but the most difficult game should be the first one on Friday night. Sandy Alcantara comes into town to face the Fightins for the first time since 2023. The Philadelphia Phillies actually have had a pretty good amount of success against the 2022 NL Cy Young Winner, and they have their own ace, Zack Wheeler toeing the rubber.

On Saturday, we get to see if Taijuan Walker can continue his success to start the season. In three starts, he has only had one inning where runs have crossed the plate; however, the Giants tagged him for 6 runs in the second inning on Monday. He also has had traffic in other innings, but I feel pretty good about him for the most part to be honest.

Jesus Luzardo gets the ball on Sunday against his former team and what a mistake it looks like from the Marlins. The Jesus Lizard has been lights out to start the season for the Phillies, and I would expect him to dominate his old club during the Easter matinee. We have Jesus pitching on Easter! Lol.

In [ ]:
similar_hits(pos
             ,po25[(po25.player_name == 'Bohm, Alec') & (po25.events == 'triple')].launch_speed.unique()[0]
             ,po25[(po25.player_name == 'Bohm, Alec') & (po25.events == 'triple')].launch_angle.unique()[0]
             ,po25[(po25.player_name == 'Bohm, Alec') & (po25.events == 'triple')].hit_distance_sc.unique()[0]
             ,po25[(po25.player_name == 'Bohm, Alec') & (po25.events == 'triple')].release_speed.unique()[0]
            )

## Game 1: Wheeler vs Alcantara

In [ ]:
# Finding Sandy's ID through Jake Cave home runs
pos[(pos.events == 'home_run') & (pos.player_name == 'Cave, Jake')][['game_date','pitcher','away_team','home_team']]
sandy = 645261
nresults('player_name',nphl[nphl.pitcher == sandy])
op = nphl[nphl.player_name == 'Alcantara, Sandy']
nresults('game_year',op)

In [ ]:
pitch_plot(op)

In [ ]:
# Sandy in his Entire Career
lpm = lhb_pitch_mix(op)
lpm['stand'] = 'L'
rpm = rhb_pitch_mix(op)
rpm['stand'] = 'R'
pm = pd.concat([lpm,rpm])
fig = px.scatter(pm[pm.pitch_type != 'PO']
                 ,x='plate_x'
                 ,y='plate_z'
                 ,color='pitch_name'
                 ,size='usage'
                 ,text='pitch_type'
                 ,title = '{} {} Pitch Mix'.format(op.player_name.unique()[0].split(', ')[1]
                                                   ,op.player_name.unique()[0].split(', ')[0]
                                                  )
                 ,facet_col = 'stand'
                 ,hover_data = ['release_speed','release_spin_rate','pfx_x','pfx_z']
                )

x_axes = [ax for ax in fig.layout if ax.startswith('xaxis')]
y_axes = [ax for ax in fig.layout if ax.startswith('yaxis')]

# Add Strike Zone
fig.add_shape(type='rect'
              ,x0=-0.83,x1=0.83
              ,y0=op.sz_bot.mean()
              ,y1=op.sz_top.mean()
              ,line=dict(color='black',width=1)
              #,xref=x_ax.replace('axis', '')
              #,yref=y_ax.replace('axis', '')
             )

fig.show()

In [ ]:
# Sandy this year
fig = px.scatter(op[op.game_year == 2025]
                 ,x='plate_x'
                 ,y='plate_z'
                 ,color='pitch_name'
                 #,size='usage'
                 #,text='pitch_type'
                 ,title = '{} {} Pitch Mix'.format(op.player_name.unique()[0].split(', ')[1]
                                                   ,op.player_name.unique()[0].split(', ')[0]
                                                  )
                 ,facet_col = 'stand'
                 ,hover_data = ['release_speed','release_spin_rate','pfx_x','pfx_z']
                )

x_axes = [ax for ax in fig.layout if ax.startswith('xaxis')]
y_axes = [ax for ax in fig.layout if ax.startswith('yaxis')]

# Add Strike Zone
fig.add_shape(type='rect'
              ,x0=-0.83,x1=0.83
              ,y0=op.sz_bot.mean()
              ,y1=op.sz_top.mean()
              ,line=dict(color='black',width=1)
              #,xref=x_ax.replace('axis', '')
              #,yref=y_ax.replace('axis', '')
             )

fig.show()

In [ ]:
# Sandy in his Entire Career
lpm = lhb_pitch_mix(op[op.game_year == 2025])
lpm['stand'] = 'L'
rpm = rhb_pitch_mix(op[op.game_year == 2025])
rpm['stand'] = 'R'
pm = pd.concat([lpm,rpm])
fig = px.scatter(pm[pm.pitch_type != 'PO']
                 ,x='plate_x'
                 ,y='plate_z'
                 ,color='pitch_name'
                 ,size='usage'
                 ,text='pitch_type'
                 ,title = '{} {} Pitch Mix'.format(op.player_name.unique()[0].split(', ')[1]
                                                   ,op.player_name.unique()[0].split(', ')[0]
                                                  )
                 ,facet_col = 'stand'
                 ,hover_data = ['release_speed','release_spin_rate','pfx_x','pfx_z']
                )

x_axes = [ax for ax in fig.layout if ax.startswith('xaxis')]
y_axes = [ax for ax in fig.layout if ax.startswith('yaxis')]

# Add Strike Zone
fig.add_shape(type='rect'
              ,x0=-0.83,x1=0.83
              ,y0=op.sz_bot.mean()
              ,y1=op.sz_top.mean()
              ,line=dict(color='black',width=1)
              #,xref=x_ax.replace('axis', '')
              #,yref=y_ax.replace('axis', '')
             )

fig.show()

In [ ]:
# It looks like he is trying to throw more CH and backfoot SL to LHB?

## Game 2: Walker vs Quantrill

## Game 3: Luzardo vs Gillispie

In [ ]:
# I will be curious to see if his pitch mix is different with Marchan vs Realmuto
# How much of that can you attribute to the catcher?
# Situations or counts?

In [ ]:
nresults('player_name',pp25)
whiff_rate('pitch_type',pps[pps.player_name == 'Luzardo, Jesús'])

In [ ]:
whiff_rate(['player_name','pitch_type'],pp25)

In [ ]:
# Is Trea Turner getting a lot of breaking balls?

In [ ]:
pitch_mix(po25[(po25.player_name == 'Turner, Trea') & (po25.p_throws == 'R')])

In [ ]:
pitch_mix(po25[(po25.player_name == 'Turner, Trea') & (po25.p_throws == 'L')])

In [ ]:
similar_hits(pd.concat([nphl,pos,pps])
             ,77,10,148,89)

In [ ]:
jl = pps[pps.player_name == 'Luzardo, Jesús']
z = jl.groupby('bb_type',as_index=False).agg(count = ('des','count'))
z['rate'] = z['count']/z['count'].sum()
z

In [ ]:
wr = whiff_rate('pitch_number',pp25[(pp25.away_team == 'MIA')])
res = nresults('pitch_number',pp25[(pp25.away_team == 'MIA')])
final = res.merge(wr, on = 'pitch_number')
final['rate'] = final.swings/final.pitches
final
# 38% of the first pitches this weekend were swung at by the Miami Marlins

In [ ]:
wr = whiff_rate('pitch_number',pp25[(pp25.away_team != 'MIA')])
res = nresults('pitch_number',pp25[(pp25.away_team != 'MIA')])
final = res.merge(wr, on = 'pitch_number')
final['rate'] = final.swings/final.pitches
final
# 31% of the first pitches in every other game get swung at

In [ ]:
test = pp25[pp25.away_team == 'MIA'].groupby(['game_pk','at_bat_number'],as_index=False).agg(pitches = ('pitch_number','max'))
test.at_bat_number.count()

In [ ]:
nresults('inning',po25)

In [ ]:
nresults('inning',pp25)

# NYM (A) 421-423

## Game 1: Nola vs Megill

In [ ]:
nresults('p_throws',pos[pos.player_name == 'Kepler, Max'])

In [ ]:
nresults('game_year',nola)

# Try to do something worthwhile bc this game sucks

In [ ]:
df = nola[['player_name'
      ,'p_throws'
      ,'stand'
      ,'balls'
      ,'strikes'
      ,'outs_when_up'
      ,'pitch_type'
      ,'release_speed'
      ,'release_spin_rate'
      ,'pfx_x'
      ,'pfx_z'
      ,'plate_x'
      ,'plate_z'
      # ,'spin_axis'
      # ,'release_pos_y'
      # ,'release_extension'
     ]]

y = df.pitch_type

In [ ]:
nresults('stand',pps[pps.player_name == 'Banks, Tanner'])

In [ ]:
# Release Position!

In [ ]:
fig = px.scatter(nola
                 ,x='release_pos_x'
                 ,y='release_pos_z'
                 ,color='pitch_name'
                 ,facet_col='game_year',facet_col_wrap=4
                )
fig.show()

In [ ]:
fig = px.scatter(pp25
                 ,x='release_pos_x'
                 ,y='release_pos_z'
                 ,color='p_throws'
                 ,hover_data=['player_name']
                 #,facet_col='game_year',facet_col_wrap=4
                )
fig.show()

In [ ]:
fig = px.scatter_3d(pp25
                    ,x='release_pos_x'
                    ,y='release_pos_y'
                    ,z='release_pos_z'
                    ,color='p_throws'
                    ,hover_data = ['player_name','release_extension']
                   )
#fig.show()

In [ ]:
pp25.groupby('player_name',as_index=False).agg(extendo = ('release_extension','mean')
                                               ,spin_axis = ('spin_axis','mean')
                                               ,release_y = ('release_pos_y', 'mean')
                                              )

In [ ]:
nresults('inning',po25)

In [ ]:
nresults('pitch_name',bohm[bohm.p_throws == 'R']).sort_values(by='pitches',ascending=False)

In [ ]:
#pos[pos.pitch_type == 'SC']
#pps[pps.pitch_type == 'SC']
nphl[nphl.pitch_type == 'SC']

## Game 2: Sanchez vs Canning

Sanchez in the first inning of games on the road, not good!

In [ ]:
away = cs[cs.home_team != 'PHI']
home = cs[cs.home_team == 'PHI']
resa = nresults('inning',away)
resa['home_team'] = 'away'
resh = nresults('inning',home)
resh['home_team'] = 'home'
res = pd.concat([resa,resh])
res['krate'] = res.strikeouts/res.plate_apps
res['bbrate'] = res.walks/res.plate_apps
res[res.inning==1]

In [ ]:
resa.merge(whiff_rate('inning',away).merge(chase_rate('inning',away))
          ,on='inning')

In [ ]:
# Does Bohm get more off speed in first pitch?
df = bohm[bohm.pitch_number == 1]
nresults('game_year',df)

In [ ]:
level = 'game_year'
z = whiff_rate(level,df).merge(chase_rate(level,df),on=level)
final = z.merge(nresults(level,df),on=level)
final['swing_rate'] = final.swings/final.pitches
fig = px.scatter(final
                 ,x='game_year'
                 ,y='swing_rate'
                 ,size='pitches'
                 ,text='swing_rate'
                )
fig.show()

In [ ]:
pitch_mix(df)

In [ ]:
fig = px.scatter(final
                 ,x='game_year'
                 ,y='whiff_rate'
                 ,size='pitches'
                 ,text='whiff_rate'
                )
fig.show()

In [ ]:
# Differentiate between SL and CH for Sanchez
pitch_plot(cs)

In [ ]:
fig = px.scatter(cs[cs.pitch_type.isin(['SL','CH'])]
                 ,x='pfx_x'
                 ,y='pfx_z'
             ,color='pitch_name'
                 ,hover_data = ['release_speed','release_spin_rate','release_extension','spin_axis']
                )
fig.show()

In [ ]:
# What percent of Bryce Harper's plate appearances with the Phillies are with runners on base?
df = pos[(pos.player_name == 'Harper, Bryce')&(pos.on_1b.isna()==True)&(pos.on_2b.isna()==True)&(pos.on_3b.isna()==True)]
res_no_runners = nresults('game_year',df)
res_all = nresults('game_year',bh)
level = 'game_year'
res_no_runners = nresults(level,df)
res_all = nresults(level,bh)
res = res_no_runners.merge(res_all,on=level,suffixes= ('_no_runners','')
                          )
res[['game_year'
     ,'pitches_no_runners','pitches'
     ,'plate_apps_no_runners','plate_apps'
     ]]
res['share'] = res.plate_apps_no_runners/res.plate_apps
res[['game_year'
     ,'pitches_no_runners','pitches'
     ,'plate_apps_no_runners','plate_apps','share'
     ]]

In [ ]:
res['At Bats with Runners On'] = round((100*res.share),1)
fig_color = 'At Bats with Runners On'
res['year'] = res.game_year.astype('str')
fig = px.scatter(res
                 ,x='game_year'
                 ,y='share'
                 ,size='pitches'
                 ,text='At Bats with Runners On'
                 ,title = 'Bryce Harper Share of {} by {}'.format(fig_color
                                                                  ,level
                                                                 )
                 ,color='year'
                )
fig.show()

In [ ]:
nc = pos[(pos.player_name == 'Castellanos, Nick')&(pos.p_throws == 'R')&(pos.pitch_type.isin(['SI','SL']))]
pitch_plot(nc)

In [ ]:
df = pos[pos.player_name == 'Castellanos, Nick']
pitch_plot(df[df.description.isin(whiffs)])

In [ ]:
res = nresults('player_name',pos[pos.game_year == 2025])
res

In [ ]:
# for p in nresults('player_name',pos[pos.game_year == 2025]).player_name:
#     pitch_plot(pos[(pos.player_name == p)&(pos.description.isin(whiffs)))

In [ ]:
rojas = pos[pos.player_name == 'Rojas, Johan']
nresults('game_year',rojas)

In [ ]:
bb = rojas[rojas.events=='walk']
bb['count'] = bb.balls.astype('str') + '-' + bb.strikes.astype('str')
bb
#nresults('count',bb)

In [ ]:
nresults('game_year',bb)

In [ ]:
z = bb.groupby(['game_year','count'],as_index=False).agg(walks = ('des','count'))
z.groupby('count',as_index=False).agg(total_walks = ('walks','sum'))

In [ ]:
# 13 of his 23 career walks have been off full counts, Kellen you ignorant slut

In [ ]:
# Fuck great catch by Kepler in the bottom of the 8th on Tuesday
gd = '2025-04-22'
z = pps[(pps.game_date == gd)&(pps.inning==8)&(pps.hit_location == 7)][['launch_speed','launch_angle'
                                                                   ,'hit_distance_sc','hc_x','hc_y','release_speed'
                                                                   ,'estimated_ba_using_speedangle']]
z

In [ ]:
similar_hits(pd.concat([nphl,pos,pps])
             ,z.launch_speed.unique()[0]
             ,z.launch_angle.unique()[0]
             ,z.hit_distance_sc.unique()[0]
             ,z.release_speed.unique()[0]
            )

In [ ]:
res = nresults('inning',po25)
res['pitches_per_plate_app'] = res.pitches/res.plate_apps
res

## Game 3: Wheeler vs Peterson

In [ ]:
# Wheeler gives up homers to scrubs
zw = pps[pps.player_name == 'Wheeler, Zack']
pps[pps.batter.isin(zw[zw.events == 'home_run'].groupby('batter',as_index=False).agg(hrs = ('des','count')
                                                                    ,ev = ('launch_speed','mean')
                                                                 ,maxev = ('launch_speed','max')
                                                                ).sort_values(by='hrs'
                                                                              ,ascending=False
                                                                             ).head(10).batter.tolist()
                   )
].groupby('batter',as_index=False).agg({'des' : 'max'})

In [ ]:
pps[pps.batter == 457763].des.tolist()[0:5]
# Buster Posey
# Travis d'Arnaud
# Marcell Ozuna
# Josh Bell
# Brandon Nimmo
# Ronald Acuna Jr
# Keibert Ruiz
# Juan Soto
# Jazz Chisholm

In [ ]:
# so not really scrubs

# CHC (A) 425-427

## Game 1: Walker vs Rea

In [ ]:
pms = pd.DataFrame()
for gy in tw.game_year.unique().tolist():
    lpm = lhb_pitch_mix(tw[tw.game_year == gy])
    lpm['stand'] = 'L'
    rpm = rhb_pitch_mix(tw[tw.game_year == gy])
    rpm['stand'] = 'R'
    pm = pd.concat([lpm,rpm])
    pm['game_year'] = gy
    pms = pd.concat([pms,pm])

pms['year'] = pms.game_year.astype('str')
z = pms

In [ ]:
fig = px.scatter(z.sort_values(by='game_year')
                 ,x='plate_x'
                 ,y='plate_z'
                 ,size='usage'
                 ,color='pitch_name'
                 ,facet_col='year'
                 ,facet_row = 'stand'
                 ,text='pitch_type'
                 ,title = 'Taijuan Walker Pitch Mix'
                )
fig.add_shape(type='rect'
              ,x0=-0.83,x1=0.83
              ,y0=tw.sz_bot.mean()
              ,y1=tw.sz_top.mean()
              ,line=dict(color='black',width=1)
             )
fig.show()

In [ ]:
# Is Taijuan Walker winning at 1-1?
df = tw[(tw.balls == 1) & (tw.strikes == 1)]
nresults('game_year',df)

In [ ]:
df.groupby('description',as_index=False).agg(occs = ('des','count'))

In [ ]:
whiff_rate('game_year',df).merge(chase_rate('game_year',df),on='game_year')

In [ ]:
df.groupby('game_year',as_index=False).agg(velo = ('release_speed','mean'))

In [ ]:
pitch_plot(df)

In [ ]:
# Yes it appears Tai Walker is winning at 1-1

In [ ]:
df

In [ ]:
z = pps.merge(df, on = ['game_pk','at_bat_number'],suffixes = ('','_11'))
z.groupby(['game_pk','at_bat_number'],as_index=False
         ).agg({'pitch_number' : 'max'})

In [ ]:
nresults('game_year',z)
# Yes after returning to the at bat after the 1-1 pitch I see that Taijuan Walker is more consistently winning his 1-1 at bats

In [ ]:
# Trea has been aggressive early in the count, he has done damage on the first pitch!
nresults('pitch_number',po25[po25.player_name == 'Turner, Trea'])

In [ ]:
pitch_mix(po25[(po25.pitch_number == 1) & (po25.player_name == 'Turner, Trea')])

In [ ]:
# Stott pitches per plate appearance
res = nresults('game_year',pos[pos.player_name == 'Stott, Bryson'])
res['pitches_per_plate_app'] = res.pitches/res.plate_apps
res

In [ ]:
# Who does Schwarber play LF behind?
nresults('player_name',pp25[pp25.fielder_7 == po25[po25.player_name == 'Schwarber, Kyle'].batter.unique()[0]]).sort_values(by='pitches',ascending=False)

In [ ]:
# Who does Schwarber play LF behind?
nresults('player_name',pps[pps.fielder_7 == po25[po25.player_name == 'Schwarber, Kyle'].batter.unique()[0]]).sort_values(by='pitches',ascending=False)

In [ ]:
res_all = nresults('player_name',pps[pps.game_year >= 2022])
res_f7 = nresults('player_name',pps[(pps.hit_location == 7)&(pps.game_year >= 2022)])
z = res_all.merge(res_f7,on='player_name',suffixes = ('','_7'))
z['share'] = z.bip_7/z.bip
z[z.player_name.isin(nresults('player_name',pp25).player_name)][['player_name','bip','bip_7','share']].sort_values(by='share',ascending=False)
# It looks like Wheeler and Sanchez see a smaller percentage of balls in play go to Left Field than other pitchers.

In [ ]:
op = nphl[nphl.player_name == 'Rea, Colin']
nresults('game_year',op)

In [ ]:
# Pitch Mix
lpm = lhb_pitch_mix(op)
lpm['stand'] = 'L'
rpm = rhb_pitch_mix(op)
rpm['stand'] = 'R'
pm = pd.concat([lpm,rpm])
fig = px.scatter(pm[~pm.pitch_type.isin(['IN','PO'])]
                 ,x='plate_x'
                 ,y='plate_z'
                 ,color='pitch_name'
                 ,size='usage'
                 ,text='pitch_type'
                 ,title = '{} {} Pitch Mix'.format(op.player_name.unique()[0].split(', ')[1]
                                                   ,op.player_name.unique()[0].split(', ')[0]
                                                  )
                 ,facet_col = 'stand'
                )

x_axes = [ax for ax in fig.layout if ax.startswith('xaxis')]
y_axes = [ax for ax in fig.layout if ax.startswith('yaxis')]

# Add Strike Zone
fig.add_shape(type='rect'
              ,x0=-0.83,x1=0.83
              ,y0=op.sz_bot.mean()
              ,y1=op.sz_top.mean()
              ,line=dict(color='black',width=1)
              #,xref=x_ax.replace('axis', '')
              #,yref=y_ax.replace('axis', '')
             )

fig.show()

In [ ]:
# Pitch Mix
lpm = lhb_pitch_mix(op[op.game_year == 2025])
lpm['stand'] = 'L'
rpm = rhb_pitch_mix(op[op.game_year == 2025])
rpm['stand'] = 'R'
pm = pd.concat([lpm,rpm])
fig = px.scatter(pm[~pm.pitch_type.isin(['IN','PO'])]
                 ,x='plate_x'
                 ,y='plate_z'
                 ,color='pitch_name'
                 ,size='usage'
                 ,text='pitch_type'
                 ,title = '{} {} Pitch Mix'.format(op.player_name.unique()[0].split(', ')[1]
                                                   ,op.player_name.unique()[0].split(', ')[0]
                                                  )
                 ,facet_col = 'stand'
                )

x_axes = [ax for ax in fig.layout if ax.startswith('xaxis')]
y_axes = [ax for ax in fig.layout if ax.startswith('yaxis')]

# Add Strike Zone
fig.add_shape(type='rect'
              ,x0=-0.83,x1=0.83
              ,y0=op.sz_bot.mean()
              ,y1=op.sz_top.mean()
              ,line=dict(color='black',width=1)
              #,xref=x_ax.replace('axis', '')
              #,yref=y_ax.replace('axis', '')
             )

fig.show()

In [ ]:
pitch_plot(op)

In [ ]:
op['count'] = op.balls.astype('str') + '-' + op.strikes.astype('str')
z = op[(op.pitch_type == 'SI') & (op.stand == 'L')]
z.groupby('strikes',as_index=False).agg(usage = ('des','count'))

In [ ]:
pm

In [ ]:
# Lineup Ingestion!
lineup = pd.DataFrame({'player_name' : ['Stott, Bryson'
                                        ,'Turner, Trea'
                                        ,'Harper, Bryce'
                                        ,'Schwarber, Kyle'
                                        ,'Castellanos, Nick'
                                        ,'Kepler, Max'
                                        ,'Realmuto, J.T.'
                                        ,'Bohm, Alec'
                                        #,'Stott, Bryson' 
                                        #,'Sosa, Edmundo'
                                        #,'Marsh, Brandon' 
                                        ,'Rojas, Johan'
                                       ]
                       ,'batting_order' : [1,2,3,4,5,6,7,8,9]
                       ,'fld_position' : [4
                                          ,6
                                          ,3
                                          ,0
                                          ,9
                                          ,7
                                          ,2
                                          ,5
                                          ,8
                                         ]
                      })
lineup
df = pd.concat([nphl[nphl.player_name.isin(lineup.player_name)]
                ,pos[pos.player_name.isin(lineup.player_name)]
               ])
nresults('player_name',df[df.pitcher == op.pitcher.unique()[0]])

In [ ]:
similar_hits(pd.concat([nphl,pps,pos])
             ,pos[(pos.player_name == 'Stott, Bryson') &(pos.events == 'double') & (pos.pitcher == op.pitcher.unique()[0])].launch_speed.unique()[0]
             ,pos[(pos.player_name == 'Stott, Bryson') &(pos.events == 'double') & (pos.pitcher == op.pitcher.unique()[0])].launch_angle.unique()[0]
             ,pos[(pos.player_name == 'Stott, Bryson') &(pos.events == 'double') & (pos.pitcher == op.pitcher.unique()[0])].hit_distance_sc.unique()[0]
             ,pos[(pos.player_name == 'Stott, Bryson') &(pos.events == 'double') & (pos.pitcher == op.pitcher.unique()[0])].release_speed.unique()[0]
            )

In [ ]:
risp = pos[(pos.on_2b.isna() == False) | (pos.on_3b.isna() == False)] # Filter to runners in scoring position
df = risp[risp.game_date >= '2025-04-20']
nresults('game_year',df)
# res = nresults('game_year',risp) # Get the results with RISP
# gy = nresults('game_year',pos) # Get the results for the entire year, used to calculate the share of plate_apps w RISP
# res = res.merge(gy,on=['game_year'],suffixes = ('','_nrisp')) # Merge the two result sets
# res['share'] = round(100*(res.plate_apps/res.plate_apps_nrisp),1) # Do the share calculation and format since it is a percentage
# res[[c for c in res.columns if 'nrisp' not in c]] # Give me only the columns I care about, this was cool!

In [ ]:
rf = df.groupby(['game_pk','at_bat_number','bat_score'
           ],as_index=False
          ).agg({'post_bat_score' : 'max'})
rf['runs_for'] = rf.post_bat_score-rf.bat_score
rf.runs_for.sum()

In [ ]:
res = nresults('game_date',pos[pos.game_date >= '2025-04-20'])
res['plate_apps_sum'] = res.plate_apps.sum()
res

In [ ]:
45/165

In [ ]:
pos[pos.game_date >= '2025-04-20'].groupby('game_date',as_index=False).agg(runs_for = ('post_bat_score','max'))

In [ ]:
df

In [ ]:
rf = pos.groupby(['game_pk','game_year','game_date','away_team','home_team'
                 ],as_index=False).agg(runs_for = ('post_bat_score','max'))
ra = pps.groupby(['game_pk','game_year','game_date','away_team','home_team'
                 ],as_index=False).agg(runs_against = ('post_bat_score','max'))
df = rf.merge(ra,on=['game_pk','game_year','game_date','away_team','home_team'])
df['wl'] = np.where(df.runs_for > df.runs_against,'W','L')
df[df.game_year==2025]

df.sort_values(by=['game_date','game_pk'],inplace=True)
# Identify where the value changes
change = df['wl'] != df['wl'].shift()

# Assign group numbers to each block of repeated values
group = change.cumsum()

# Group by those blocks and compute sizes
result = df.groupby(group)['wl'].agg(['first', 'size']).reset_index(drop=True)
result.rename(columns = {'first' : 'result'
                         ,'size' : 'streak'
                        }
              ,inplace=True
             )
result

In [ ]:
result[(result['result'] == 'L') & (result.streak == 5)]

In [ ]:
dh = pos.groupby(['game_date'],as_index=False).agg(games = ('game_pk','nunique'))
df[df.game_date.isin(dh[dh.games>1].game_date.tolist())].tail(10)

## Game 2: Luzardo vs Brown

## Game 3: Nola vs Taillon
Nola pitch plot to the glove side

In [ ]:
pitch_plot(nola[nola.plate_x >= 0])

In [ ]:
fig = px.scatter(nola
                 ,x='plate_x'
                 ,y='plate_z'
                 ,color='zone'
                )
fig.show()

In [ ]:
nresults('game_year',nola[nola.plate_x >=0])
# Nola can usually control the outside part of the plate but has struggled out there this season

In [ ]:
# does Nola pitch well at Wrigley?
res = nresults('home_team',nola)
res['krate'] = res.strikeouts/res.plate_apps
res['bbrate'] = res.walks/res.plate_apps
res_mu = nresults('player_name',nola)
res_mu['krate'] = res_mu.strikeouts/res_mu.plate_apps
res_mu['bbrate'] = res_mu.walks/res_mu.plate_apps
res_mu.rename(columns = {'player_name' : 'home_team'},inplace=True)
res_concat = pd.concat([res,res_mu])
res_concat[res_concat.home_team.isin(['CHC','PHI','Nola, Aaron'])]
# To be honest, not so much.

In [ ]:
# Nola starts in Wrigley
nres = nresults('game_date',nola[nola.home_team == 'CHC']
        ).merge(nola.groupby(['game_date','game_pk','game_year'
             ],as_index=False
            ).agg(innings = ('inning','max')
                  ,runs_allowed = ('post_bat_score','max')
                  ,horiz = ('x_break','mean')
                 ),on='game_date')
nres

In [ ]:
# Merged in Nola average performance to get the horizontal break
nres.merge(nola.groupby('game_year',as_index=False).agg(horiz = ('x_break','mean'))
           ,on='game_year'
           ,suffixes = ('','_mu')
          )

In [ ]:
# Game Results for starts by Aaron Nola with at least four walks
gms = nresults('game_date',nola)
gms[gms.walks >=4].sort_values(by=['walks','game_date'],ascending=False)

In [ ]:
# Nola 2025 starts
gms['game_year'] = pd.to_datetime(gms.game_date).dt.year
gms25 = gms[gms.game_year == 2025].merge(nola.groupby(['game_date','game_pk','game_year'
             ],as_index=False
            ).agg(innings = ('inning','max')
                  ,runs_allowed = ('post_bat_score','max')
                  ,horiz = ('x_break','mean')
                 ),on=['game_date','game_year'],suffixes=('','_nola')
                                )
gms25

In [ ]:
# Nola Pitch Mix by Game
pms = pd.DataFrame()
for gm in gms25.game_date:
    lpm = lhb_pitch_mix(nola[nola.game_date == gm])
    lpm['stand'] = 'L'
    rpm = rhb_pitch_mix(nola[nola.game_date == gm])
    rpm['stand'] = 'R'
    pm = pd.concat([lpm,rpm])
    pm['game_date'] = gm
    pms = pd.concat([pms,pm])

fig = px.scatter(pms.sort_values(by=['game_date','stand'])
                 ,x='plate_x'
                 ,y='plate_z'
                 ,color='pitch_name'
                 ,size='usage'
                 ,text='pitch_type'
                 ,facet_row = 'game_date'
                 ,facet_col = 'stand'
                )
fig.show()

#### Taillon

In [ ]:
op = nphl[nphl.player_name == 'Taillon, Jameson']
nresults('game_year',op)

In [ ]:
pitch_plot(op)

In [ ]:
nresults('player_name',op) # Jameson Taillon is at 999 career strikeouts

In [ ]:
# Has he ever pitched in the playoffs? Yes, he has faced 23 batters in the playoffs and recorded 0 strikeouts.
nresults('game_type',op)

In [ ]:
nresults('game_pk',op[op.game_type != 'R'])

In [ ]:
nphl[nphl.game_date == '2022-10-14'].groupby(['game_pk','away_team','home_team']
                                            ,as_index=False).agg(players = ('player_name','nunique'))

In [ ]:
nphl[nphl.game_date == '2022-10-19'].groupby(['game_pk','away_team','home_team']
                                            ,as_index=False).agg(players = ('player_name','nunique'))

#### Stott pitches per AB and 2-strike hits

In [ ]:
nresults('strikes',po25[po25.player_name == 'Stott, Bryson']) # 17 hits with 2 strikes, the second highest in baseball

In [ ]:
res = nresults('player_name',po25[po25.player_name == 'Stott, Bryson'])
res['pitches_per_pa'] = res.pitches/res.plate_apps
res # At 4.625, Bryson Stott sees the most pitches per plate_app in the league

In [ ]:
lhb_pitch_mix(op[(op.balls == 2) & (op.strikes == 0)])

#### Matt Strahm Release Point

In [ ]:
nresults('player_name',pp25[pp25.p_throws == 'L'])

In [ ]:
# Matt Strahm Release Point by Stand
fig = px.scatter(pps[pps.player_name.isin(nresults('player_name',pp25[pp25.p_throws=='L']).player_name.tolist())]
                 ,x='release_pos_x'
                 ,y='release_pos_z'
                 ,color='stand'
                 ,facet_row = 'player_name'
                )
fig.show()

In [ ]:
# How long has Matt Strahm been doing this?
fig = px.scatter(pd.concat([pps[pps.player_name.isin(['Strahm, Matt'])]
                           ,nphl[nphl.player_name == 'Strahm, Matt']]).sort_values(by='game_year')
                 ,x='release_pos_x'
                 ,y='release_pos_z'
                 ,color='stand'
                 ,facet_col = 'game_year',facet_col_wrap=4
                 ,title = 'Matt Strahm Release Point over Time'
                )
fig.show()

In [ ]:
jt = pos[pos.player_name == 'Realmuto, J.T.']
nresults('inning',jt) # generally pretty good in Extra Innings

#### Kepler Top 10

In [ ]:
mk = pd.concat([nphl[nphl.player_name == 'Kepler, Max']
                ,pos[pos.player_name == 'Kepler, Max']
               ])
#nresults('game_year',mk[(mk.balls == 2) & (mk.strikes == 0)])
df = mk[(mk.balls == 2) & (mk.strikes == 0)]
test = df.groupby(['game_pk','at_bat_number'
           ],as_index=False).agg(events = ('events','max')
                                 ,types = ('type','max')
                                 ,pitches = ('pitch_number','max')
                                )
test.groupby('events',as_index=False).agg(counter = ('types','count'))

In [ ]:
gpd = mk.groupby(['game_pk','at_bat_number','events'],as_index=False
          ).agg(pitches = ('pitch_number','max'))

z = gpd.merge(df.groupby(['game_pk','at_bat_number'],as_index=False).agg(test = ('events','max'))
          ,on=['game_pk','at_bat_number']
          ,suffixes = ('','_20')
         )
zz = z.groupby('events',as_index=False).agg(counter = ('at_bat_number','count'))
wins = ['double','hit_by_pitch','field_error','fielders_choice','home_run','intent_walk','sac_fly','single','triple','walk']
win_rate = zz[zz.events.isin(wins)].counter.sum()/zz.counter.sum()
win_rate # Kepler wins the at bat 55% of the time when it gets to 0-2

#### Bohm Top 10 with Bases Loaded

In [ ]:
# Nearly hits a grand slam!
similar_hits(pd.concat([pos[pos.stand == 'R']
                        ,nphl[nphl.stand == 'R']
                        ,pps[pps.stand == 'R']
                       ])
             ,102
             ,28
             ,350
             ,94.2)

In [ ]:
# Stott draws a walk, tough.

In [ ]:
# Trea Turner now stepping to the plate with the chance to deliver a victory
# He does so with his speed lol, so one run scores

In [ ]:
similar_hits(pd.concat([pos[pos.player_name == 'Turner, Trea']
                        ,nphl[nphl.player_name == 'Turner, Trea']
                       ])
             ,83.7
             ,-23
             ,6
             ,83.9)

In [ ]:
df = pos[pos.events == 'home_run'].groupby(['game_date'
                                       ,'game_pk'
                                       ,'away_team'
                                       ,'home_team'
                                       ,'player_name'
                                      ],as_index=False
                                     ).agg(hrs = ('des','count')
                                          )
df[df.hrs > 1]

In [ ]:
sh = pos.groupby('player_name',as_index=False).agg(stands = ('stand','nunique'))
sh[sh.stands > 1].merge(df[df.hrs >1 ], on = 'player_name')

# March and April Recap

In [ ]:
# Hardest and Furthest Hit - Kyle Schwarber off Chris Sales in ATL
po25[['player_name','events','inning','game_date'
     ,'away_team','home_team'
     ,'launch_speed','launch_angle','hit_distance_sc']].sort_values(by='launch_speed',ascending=False).head(5)

In [ ]:
# Most Win Probability Added - Bohm double on Opening Day
po25[['player_name','events','inning','game_date'
     ,'away_team','home_team'
     ,'launch_speed','launch_angle','hit_distance_sc','wpa']].sort_values(by='wpa',ascending=False).head(5)

In [ ]:
# Most Win Probability Added - Bohm keeps the winning run at third in extras down in Atlanta
pp25[['player_name','events','inning','game_date'
     ,'away_team','home_team'
     ,'launch_speed','launch_angle','hit_distance_sc','wpa']].sort_values(by='wpa',ascending=False).head(15)

In [ ]:
# Fastest Pitch - Alvarado two at 101.6 in ATL
# Most Spin - Kerkering called strike to Toglia
# Most Horizontal Movement - Nola punches out Adames
pp25[['player_name','events','inning','game_date','description'
     ,'away_team','home_team','pitch_type'
     ,'release_speed','release_spin_rate','pfx_x','pfx_z'
     ,'x_break','z_break']].sort_values(by='x_break',ascending=False).head(5)

In [ ]:
# I could look at Bohm's hard hit outs
bohm25 = bohm[bohm.game_year == 2025]
bohm25[bohm25.launch_speed >= 100]

In [ ]:
# Alvarado struck out Xavier Edwards on three pitches
pp25.groupby(['player_name','pitcher'],as_index=False).agg(pitches = ('des','count'))
ja = pps[pps.player_name == 'Alvarado, José']
ks = ja[ja.events.isin(['strikeout','strikeout_double_play'])].groupby(['game_pk','at_bat_number'],as_index=False).agg({'des' : 'count'})
pabs = pps.groupby(['game_date','away_team','home_team','inning','game_pk','at_bat_number'],as_index=False).agg(pitches = ('des','count'))
pabs[pabs.pitches == 3].merge(ks,on=['game_pk','at_bat_number'],suffixes=('','_ks'))

In [ ]:
nresults('player_name',pp25)

In [ ]:
# Alvy has been lights out and has a strike 1, strike 2, good luck style punchout
# Tanner Banks has been Tanner Banks but has pretty severe reverse splits
tb = pd.concat([nphl[nphl.player_name == 'Banks, Tanner']
                ,pps[pps.player_name == 'Banks, Tanner']
               ])
res = nresults('game_year',tb)
res['krate'] = res.strikeouts/res.plate_apps
res['bbrate'] = res.walks/res.plate_apps
res

In [ ]:
splits = pd.DataFrame()
for gy in tb.game_year.unique().tolist():
    res = nresults('stand',tb[tb.game_year == gy])
    res['game_year'] = gy
    splits = pd.concat([splits,res])
splits.sort_values(by=['game_year','stand'])

In [ ]:
# Carlos Hernandez has a pulse
df = ch = pd.concat([nphl[nphl.player_name == 'Hernández, Carlos']
                     ,pps[pps.player_name == 'Hernández, Carlos']
                    ])
nresults('game_year',df)

In [ ]:
# Orion Kerkering has struggled, but maybe look at the types of games he gets put into
df = ok = pps[pps.player_name == 'Kerkering, Orion']
nresults('game_year',df)

In [ ]:
# Luzardo has been lights out

In [ ]:
# Nola wil be ok

In [ ]:
# Romano inconsistent velocity but it seems to be getting better
fig = px.box(pp25[(pp25.player_name == 'Romano, Jordan') & (pp25.pitch_type == 'FF')].sort_values(by='game_date')
             ,x='game_date'
             ,y='release_speed'
             ,title = 'Jordan Romano Fastball Velocities by Start'
            )
fig.show()

In [ ]:
# Joe Ross - provided some length
# Ruiz - not been very good
# Strahm - just a solid solid guy
# Sanchez - little bumpy but quite good to get things going
# Walker - I have been impressed
# Wheeler - He has been good enough

## Offense

In [ ]:
nresults('player_name',po25)
# Bohm is ass but hits the ball hard
# Casty has been pretty good but is starting to slow down
# Kody Clemens is gone
# Bryce Harper is good, can be better and most likely will be better
# Max Kepler has felt up and down but has generally been positive and has strong underlying numbers
# Rafael Marchan has not done much at the plate
# Brandon Marsh got himself either hurt or demoted or both and is kind of a mess
# JT is washed
# Rojas has been able to get some hits but offers very little power
# Schwarber has been excellent, drawing lots of walks, hitting for power, and being a really good baseball player
# Edmundo was red hot and has cooled off a bit
# Cal Stevenson exists
# Bryson Stott has done well in the leadoff spot, seeing lots of pitches
# Trea Turner has walked more this year than before but is not driving the ball
# Wes Wilson is here to save the day

In [ ]:
xbh = ['double','triple','home_run']
po25[po25.events.isin(xbh)].groupby(['player_name'
                                    ],as_index=False
                                   ).agg(xbhs = ('des','count')
                                         ,ev = ('launch_speed','mean')
                                         ,la = ('launch_angle','mean')
                                         ,dist = ('hit_distance_sc','mean')
                                        ).sort_values(by='xbhs',ascending=False).round(1)

In [ ]:
level = 'player_name'
df = po25[po25.type=='X']
#inds('player_name',po25)
calcs = df.groupby(level,as_index=False).agg(counter = ('des','count')
                                                         ,unique_players = ('batter','nunique')
                                                         ,pitch_speed_mu = ('release_speed', 'mean')
                                                         ,ev_mu = ('launch_speed','mean')
                                             ,ev_std = ('launch_speed','std')
                                                         ,la_mu = ('launch_angle','mean')
                                                         ,dist_mu = ('hit_distance_sc','mean')
                                                        )
calcs.round(1).merge(df.groupby('player_name',as_index=False).agg(xba = ('estimated_ba_using_speedangle','mean')).round(3)
                     ,on='player_name'
                    ).sort_values(by='counter',ascending=False)

In [ ]:
# There may be some reason to believe that JT Realmuto will bounce back
# Maybe he has just been unlucky

In [ ]:
hard_hits('game_year',jt)

In [ ]:
hard_hits('game_year',mk) # Kepler's highest hard hit rate since 2023

In [ ]:
hard_hits('game_year',pd.concat([nphl[nphl.player_name == 'Castellanos, Nick'],pos[pos.player_name == 'Castellanos, Nick']]))

# WAS (H) 429-51
Can the Phillies drop the hammer on the Nats?

## Game 1: Wheeler vs Gore
A rematch from Opening Day when these two both dealt. Hitters battled shadows but we should expect good things here.
<br> Damn this was high-key a crazy game and I very high-key missed it.

In [ ]:
# Wheeler fastball velocity was up this start

In [ ]:
zw25 = zw[zw.game_year == 2025]
fig = px.scatter(zw25.sort_values(by='game_date')
                 ,x='plate_x'
                 ,y='plate_z'
                 ,color = 'pitch_name'
                 ,facet_col = 'game_date'
                 ,facet_row = 'stand'
                 ,title = 'Zack Wheeler Pitch Plots by Start in 2025'
                )
fig.show()

In [ ]:
a = zw.groupby(['game_pk'
            ,'game_date'
            ,'away_team'
            ,'home_team'
           ],as_index=False
          ).agg(max_inning = ('inning','max')
               )
b = pps.groupby(['game_pk'
                 ,'game_date'
                 ,'away_team'
                 ,'home_team'
                 ,'inning'
                ],as_index=False
               ).agg(pitchers = ('pitcher','nunique')
                    )
c = a.merge(b
            ,left_on = ['game_pk'
                        ,'game_date'
                        ,'away_team'
                        ,'home_team'
                        ,'max_inning'
                       ]
            ,right_on = ['game_pk'
                         ,'game_date'
                         ,'away_team'
                         ,'home_team'
                         ,'inning'
                        ]
            ,suffixes = ('','_b')
           )
c[c.pitchers > 1]

## Game 2: Sanchez vs Irvin

In [ ]:
# Kind of an easy win, team.

## Game 3: Walker vs Lord

In [ ]:
pos[(pos.launch_speed == 93.1) & (pos.launch_angle == 3)]

In [ ]:
tw25 = tw[tw.game_year == 2025]
nresults('stand',tw25)

In [ ]:
fig = px.scatter(pps[(pps.stand != pps.p_throws) & (pps.p_throws == 'R') & (pps.pitch_type == 'FC') & (pps.plate_x < 0 )]
                 ,x='plate_x'
                 ,y='plate_z'
                 ,color='zone'
                )
fig.show()

In [ ]:
pitch_mix_by_group(pps[(pps.stand != pps.p_throws) & (pps.p_throws == 'R') & (pps.pitch_type == 'FC') & (pps.plate_x < 0 )],'zone')
# Righties tend to throw backdoor cutters up in the zone

In [ ]:
pitch_mix_by_group(pps[(pps.stand != pps.p_throws) & (pps.p_throws == 'L') & (pps.pitch_type == 'FC') & (pps.plate_x > 0 )],'zone')
# Righties tend to throw backdoor cutters up in the zone

In [ ]:
lpm = lhb_pitch_mix(tw25)
lpm

In [ ]:
lpms = pd.DataFrame() 
for gy in tw.game_year.unique().tolist():
    lpm = lhb_pitch_mix(tw[tw.game_year == gy])
    lpm['game_year'] = gy
    lpms = pd.concat([lpms,lpm])

lpms[lpms.pitch_type == 'FC'].sort_values(by='game_year',ascending=False)

In [ ]:
lpms.sort_values(by=['pitch_type','game_year'],ascending=False)

In [ ]:
nresults('player_name',po25)

In [ ]:
rm = pos[pos.player_name == 'Marchán, Rafael']
df = rm25 = rm[rm.game_year == 2025]
inds('player_name',df)

In [ ]:
def inds(level,df):
    calcs = df.groupby(level,as_index=False).agg(counter = ('des','count')
                                                             ,unique_players = ('batter','nunique')
                                                             ,pitch_speed_mu = ('release_speed', 'mean')
                                                             ,ev_mu = ('launch_speed','mean')
                                                 ,ev_std = ('launch_speed','std')
                                                             ,la_mu = ('launch_angle','mean')
                                                             ,dist_mu = ('hit_distance_sc','mean')
                                                            )
    return calcs.round(1)
inds('player_name',df)

In [ ]:
op = nphl[nphl.player_name == 'Lord, Brad']
# Lineup Ingestion!
lineup = pd.DataFrame({'player_name' : ['Stott, Bryson'
                                        ,'Turner, Trea'
                                        ,'Harper, Bryce'
                                        ,'Schwarber, Kyle'
                                        ,'Castellanos, Nick'
                                        ,'Kepler, Max'
                                        ,'Bohm, Alec' 
                                        ,'Rojas, Johan'
                                        ,'Marchan, Rafael'
                                       ]
                       ,'batting_order' : [1,2,3,4,5,6,7,8,9]
                       ,'fld_position' : [4
                                          ,6
                                          ,3
                                          ,0
                                          ,9
                                          ,7
                                          ,5
                                          ,8
                                          ,2
                                         ]
                      })
lineup
df = pd.concat([nphl[nphl.player_name.isin(lineup.player_name)]
                ,pos[pos.player_name.isin(lineup.player_name)]
               ])
nresults('player_name',df[df.pitcher == op.pitcher.unique()[0]])

In [ ]:
pitch_plot(op)

In [ ]:
# Taijuan Walker insists on throwing these armside cutters to LHB
# Now he has twice tried to throw a front door SI at 1-1

In [ ]:
bdc = tw[(tw.stand == 'L') & (tw.pitch_type == 'FC') & (tw.plate_x < 0)]
nresults('game_year',bdc)

In [ ]:
fig = px.scatter(bdc.sort_values(by='game_year')
                 ,x='pfx_x'
                 ,y='pfx_z'
                 ,color='release_speed'
                 ,facet_row = 'game_year'
                )
fig.show()
# Ok I think this one was actually kinda interesting, he is gtting better finish on his back door cutters!

In [ ]:
# big error by Bryce Harper
# this is a tricky one bc how do you track it?
# The ball went right by him into the RF corner

In [ ]:
# Is it time to talk about Rafael Marchan?

In [ ]:
# The defense implodes in the top of the 6th and the Phillies need to start hitting to sweep the Nats

#### Bot 6
Turner lead off single
<br> Harper flips one into right center and Turner does not go first to third
<br> Ferrer in to face Schwarber
<br> Schwarber and Castellanos bounce two balls off of Ferrer, Nick reaches and Turner scores. Silly.
<br> Do they pinch hit for Kepler in this spot?

In [ ]:
pos[pos.des.str.contains('Ferrer')].groupby('pitcher',as_index=False).agg(games = ('game_pk','nunique'))
# this did not return anything for me during the game but I think actually will in the future.

In [ ]:
df = nphl[nphl.player_name == 'Ferrer, Jose A.']
rpm = rhb_pitch_mix(df)
fig = px.scatter(rpm
                 ,x='plate_x'
                 ,y='plate_z'
                 ,color='pitch_name'
                 ,size='usage'
                 ,text='pitch_type'
                )
fig.add_shape(type='rect'
              ,x0=-0.83,x1=0.83
              ,y0=df.sz_bot.mean(),y1=df.sz_top.mean()
              ,line=dict(color='black',width=1)
             )
fig.show()

In [ ]:
# Ferrer punches out Kepler and Bohm

#### Top 7

In [ ]:
jr = pp25[pp25.player_name == 'Ross, Joe']
nresults('game_date',jr)

#### Bot 7
Can't expect much from the bottom of this order

In [ ]:
# Ferrer stays in, he will see two righties to start

In [ ]:
op = nphl[nphl.player_name == 'Ferrer, Jose A.']
nresults('player_name',pos[(pos.player_name.isin(lineup.player_name)) & (pos.pitcher == op.pitcher.unique()[0])])

#### Top 8
Joe Ross has lost the zone. He figured it out.

#### Bot 8
Can these guys spur a comeback?
<br> What should I eat?

In [ ]:
# Trea leads us off with a hit and Harper takes a big whiff at a first pitch breaking ball

In [ ]:
nresults('player_name',po25)

In [ ]:
# He fails, so now we have Schwarber

In [ ]:
# he draws a walk with a little help from Brock Ballou!

In [ ]:
# don't worry though, Casty will kill this with a GIDP

# Phillies and the GIDP

In [ ]:
res = nresults('game_year',pos[pos.events == 'grounded_into_double_play'])
res.rename(columns = {'pitches' : 'gidp'},inplace=True)
a = nresults('game_year',pos)
df = res.merge(a,on='game_year',suffixes= ('_res',''))[['game_year','gidp','plate_apps']]
df

In [ ]:
# Carlos Hernandez has reintroduced his KC to LHB and has redefnined himself in Philly